# 03 — Mars connected regions and neutral signatures

**Purpose:** turn the prepared 1° and 10° Mars cells into connected, internally similar regions; discover recurring neutral region types; detect unusual regions; and export multivariate signatures that can later be associated with processes using independent scientific evidence.

This notebook deliberately separates four statements:

1. **Measured region:** neighboring cells have similar measured properties.
2. **Neutral type:** several regions share a recurring property combination.
3. **Process hypothesis:** papers say a particular combination *could* be produced by a process and state what additional evidence is required.
4. **Validated process signature:** multiple independently validated examples define a joint distribution suitable for Earth matching.

This notebook completes the first two statements and prepares the data structure for the fourth. It does **not** turn mineral abundance, dust, thermal inertia, or neutron counts into process labels.


## What to inspect after the run

The useful outputs are:

- The connected-region boundary map: did segmentation create coherent areas rather than scattered pixels?
- The model-comparison plot: are cluster types stable across geographic holdouts?
- The neutral-type map and profile heatmap: what combinations were discovered without process labels?
- The artifact and feature-ablation plots: do coverage stripes or one correlated measurement family control the answer?
- The anomaly map: which regions are unusual and deserve inspection?
- The signature-richness map: where do we have enough measured properties to construct useful multivariate distributions?
- The literature evidence audit: which process hypotheses can these global products support, and which required evidence is still absent?

CPUs are sufficient. A GPU does not improve the methods used here.


In [ ]:
from __future__ import annotations

import importlib.metadata
import json
import math
import re
import warnings
from collections import defaultdict
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

from packaging.version import Version

REQUIRED_DISTRIBUTIONS = {
    "pyarrow": Version("14"),
    "rasterio": Version("1.3"),
    "scikit-learn": Version("1.3"),
}
missing_or_old: list[str] = []
for distribution, minimum_version in REQUIRED_DISTRIBUTIONS.items():
    try:
        installed_version = Version(importlib.metadata.version(distribution))
    except importlib.metadata.PackageNotFoundError:
        missing_or_old.append(f"{distribution}>={minimum_version} (missing)")
        continue
    if installed_version < minimum_version:
        missing_or_old.append(f"{distribution}>={minimum_version} (found {installed_version})")
if missing_or_old:
    raise RuntimeError(
        "This notebook does not install packages automatically. Use a Kaggle image containing: "
        + ", ".join(missing_or_old)
    )

In [ ]:
import joblib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import Markdown, display
from rasterio.features import shapes as raster_shapes
from rasterio.transform import from_origin
from scipy import ndimage
from sklearn.cluster import HDBSCAN, KMeans
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import adjusted_rand_score, balanced_accuracy_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

sns.set_theme(context="notebook", style="whitegrid", palette="colorblind")
pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)


@dataclass(frozen=True)
class DiscoveryConfig:
    input_root_override: str | None = None
    mars_radius_km: float = 3396.0
    held_out_fold: int = 4
    minimum_group_coverage: float = 0.98
    medium_segmentation_prototypes: int = 12
    coarse_segmentation_prototypes: int = 6
    medium_smoothing_cells: int = 5
    coarse_smoothing_cells: int = 1
    minimum_medium_region_cells: int = 4
    minimum_coarse_region_cells: int = 1
    small_region_merge_distance_quantile: float = 0.98
    small_region_merge_max_distance: float = 2.50
    small_region_merge_rounds: int = 12
    quality_residualization_alpha: float = 25.0
    maximum_cluster_count: int = 10
    maximum_gmm_cluster_count: int = 8
    maximum_subtype_cluster_count: int = 6
    minimum_subtype_regions: int = 40
    minimum_subtype_regions_per_cluster: int = 10
    minimum_subtype_stability_ari: float = 0.60
    minimum_subtype_silhouette: float = 0.20
    minimum_subtype_assigned_fraction: float = 0.60
    maximum_fragmented_support_fraction: float = 0.20
    maximum_posthoc_coverage_fraction: float = 0.10
    minimum_partition_regions: int = 12
    minimum_broad_regions_per_cluster: int = 5
    maximum_hdbscan_regions: int = 20_000
    maximum_silhouette_rows: int = 5_000
    anomaly_fraction: float = 0.02
    artifact_warning_balanced_accuracy: float = 0.65
    random_seed: int = 42


CFG = DiscoveryConfig()
RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
RNG = np.random.default_rng(CFG.random_seed)

KAGGLE_WORKING = Path("/kaggle/working")
OUTPUT_ROOT = KAGGLE_WORKING / "mars_region_discovery" if KAGGLE_WORKING.exists() else Path.cwd() / "outputs" / "mars_region_discovery"
REGION_DIR = OUTPUT_ROOT / "regions"
REPORT_DIR = OUTPUT_ROOT / "reports"
MODEL_DIR = OUTPUT_ROOT / "models"
for directory in (OUTPUT_ROOT, REGION_DIR, REPORT_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"scikit-learn: {sklearn.__version__}")
print(f"Output root: {OUTPUT_ROOT}")

## Locate and validate notebook 01 outputs

In Kaggle, save notebook 01 as a version with outputs and add that version as an input to this notebook. The loader also accepts the same live `/kaggle/working` session and the repository's local `outputs` directory.

In [ ]:
def discover_preparation_root(override: str | None = None) -> Path:
    required_relative_paths = [
        Path("processed/mars_cells_input.parquet"),
        Path("reports/feature_schema.csv"),
        Path("reports/run_summary.json"),
    ]
    candidates: list[Path] = []
    if override:
        candidates.append(Path(override).expanduser())
    candidates.extend(
        [
            Path("/kaggle/working/mars_data_preparation"),
            Path.cwd() / "outputs" / "mars_data_preparation",
        ]
    )
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for cells_path in kaggle_input.rglob("mars_cells_input.parquet"):
            if cells_path.parent.name == "processed":
                candidates.append(cells_path.parent.parent)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.resolve()
        checked.append(str(candidate))
        if all((candidate / relative).exists() for relative in required_relative_paths):
            return candidate
    raise FileNotFoundError(
        "Could not find notebook 01 outputs. Attach the saved Kaggle output or set "
        f"DiscoveryConfig.input_root_override. Checked: {checked}"
    )


INPUT_ROOT = discover_preparation_root(CFG.input_root_override)
CELLS_PATH = INPUT_ROOT / "processed" / "mars_cells_input.parquet"
FEATURE_SCHEMA_PATH = INPUT_ROOT / "reports" / "feature_schema.csv"
PREPARATION_SUMMARY_PATH = INPUT_ROOT / "reports" / "run_summary.json"

cells_df = pd.read_parquet(CELLS_PATH)
feature_schema_df = pd.read_csv(FEATURE_SCHEMA_PATH)
preparation_summary = json.loads(PREPARATION_SUMMARY_PATH.read_text(encoding="utf-8"))

required_cell_columns = {
    "cell_id",
    "planet",
    "scale",
    "row",
    "column",
    "latitude",
    "longitude",
    "latitude_min",
    "latitude_max",
    "longitude_min",
    "longitude_max",
    "spatial_fold",
    "analysis_cohort",
}
missing_cell_columns = sorted(required_cell_columns - set(cells_df.columns))
if missing_cell_columns:
    raise ValueError(f"Prepared cell table is missing required columns: {missing_cell_columns}")
if cells_df["cell_id"].duplicated().any():
    raise ValueError("Prepared cell identifiers are not unique")
if set(cells_df["planet"].dropna().unique()) != {"Mars"}:
    raise ValueError("This notebook accepts Mars rows only")
if set(cells_df["scale"].dropna().unique()) != {"medium_1deg", "coarse_10deg"}:
    raise ValueError(f"Unexpected scales: {sorted(cells_df['scale'].dropna().unique())}")

if feature_schema_df["active_for_model"].dtype != bool:
    feature_schema_df["active_for_model"] = feature_schema_df["active_for_model"].astype(str).str.lower().eq("true")
active_schema_df = feature_schema_df.loc[feature_schema_df["active_for_model"]].copy()
expected_model_columns = {f"model__{feature}" for feature in active_schema_df["feature"]}
missing_model_columns = sorted(expected_model_columns - set(cells_df.columns))
if missing_model_columns:
    raise ValueError(f"Prepared table is missing active model columns: {missing_model_columns[:10]}")

scale_counts = cells_df.groupby(["scale", "analysis_cohort"], observed=True).size().rename("cell_count").reset_index()
print(f"Input root: {INPUT_ROOT}")
display(scale_counts)

## Build comparable feature matrices

Cells are never compared across scales or incompatible coverage cohorts. Within a cohort, each science feature must be available for at least 98% of cells; one missing feature no longer discards the rest of its science family. A few remaining gaps are filled with that cohort's median, while a completely absent measurement is never manufactured. Cohorts with usable features from only one family remain descriptive-only, and intrinsically fragmented coverage is not forced into geological clusters.

In [ ]:
def grid_shape(frame: pd.DataFrame) -> tuple[int, int]:
    rows = int(frame["row"].max()) + 1
    columns = int(frame["column"].max()) + 1
    if len(frame) != rows * columns:
        raise ValueError(f"{frame['scale'].iloc[0]} is not a complete rectangular grid: {len(frame)} rows vs {rows}×{columns}")
    if frame[["row", "column"]].duplicated().any():
        raise ValueError(f"{frame['scale'].iloc[0]} contains duplicate grid positions")
    return rows, columns


def select_partition_features(
    frame: pd.DataFrame,
    scale_name: str,
    schema: pd.DataFrame,
    minimum_group_coverage: float,
) -> tuple[list[str], list[str], dict[str, float]]:
    scale_schema = schema.loc[(schema["scale"] == scale_name) & schema["active_for_model"]]
    selected_features: list[str] = []
    selected_groups: list[str] = []
    group_coverages: dict[str, float] = {}
    for group_name, group_schema in scale_schema.groupby("group", observed=True):
        features = group_schema["feature"].tolist()
        model_columns = [f"model__{feature}" for feature in features]
        coverages = frame[model_columns].notna().mean()
        group_coverages[str(group_name)] = float(coverages.min())
        usable_features = [
            feature
            for feature, model_column in zip(features, model_columns)
            if float(coverages[model_column]) >= minimum_group_coverage
        ]
        if usable_features:
            selected_features.extend(usable_features)
            selected_groups.append(str(group_name))
    if len(selected_features) < 2:
        raise ValueError(
            f"{scale_name}/{frame['analysis_cohort'].iloc[0]} has only {len(selected_features)} features "
            f"from {len(selected_groups)} usable science groups; at least two features are required"
        )
    return selected_features, selected_groups, group_coverages


def prepare_partition_matrix(frame: pd.DataFrame, features: list[str]) -> tuple[np.ndarray, dict[str, float]]:
    columns = [f"model__{feature}" for feature in features]
    values = frame[columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    imputer = SimpleImputer(strategy="median")
    matrix = imputer.fit_transform(values)
    if matrix.shape[1] != len(columns):
        raise ValueError("A selected model feature contained no finite values despite its coverage check")
    matrix = np.clip(matrix, -8.0, 8.0).astype(np.float32)
    medians = {feature: float(value) for feature, value in zip(features, imputer.statistics_)}
    return matrix, medians


def smooth_partition_matrix(
    matrix: np.ndarray,
    rows: np.ndarray,
    columns: np.ndarray,
    shape: tuple[int, int],
    window_cells: int,
) -> np.ndarray:
    if window_cells <= 1:
        return matrix.copy()
    mask = np.zeros(shape, dtype=np.float32)
    mask[rows, columns] = 1.0
    denominator = ndimage.uniform_filter(mask, size=window_cells, mode=("nearest", "wrap"))
    smoothed = np.empty_like(matrix)
    for feature_index in range(matrix.shape[1]):
        grid = np.zeros(shape, dtype=np.float32)
        grid[rows, columns] = matrix[:, feature_index]
        numerator = ndimage.uniform_filter(grid * mask, size=window_cells, mode=("nearest", "wrap"))
        smoothed[:, feature_index] = np.divide(
            numerator[rows, columns],
            denominator[rows, columns],
            out=matrix[:, feature_index].copy(),
            where=denominator[rows, columns] > 0,
        )
    return smoothed

## Segment the planet into connected homogeneous regions

For each scale/cohort, area-weighted K-means first creates a compact set of **property prototypes**. The prototype map is then split into four-neighbor connected components, with longitude wrapping across the ±180° seam. Tiny adjacent components are iteratively merged into the most similar neighbor only when their feature-space distance is within the normal range of adjacent-region distances. This use of K-means is only a neutral quantization step: final region types are discovered later from region summaries.

A 5×5 masked smoothing window is used only for the 1° segmentation boundaries. It reduces one-cell noise without mixing coverage cohorts or changing the physical values stored for each region. Prototype counts are reduced for sparse or low-dimensional cohorts, and adjacent tiny components receive a wider—but still bounded—merge tolerance. Original unsmoothed physical measurements are retained for every summary and signature.

In [ ]:
class DisjointSet:
    def __init__(self, size: int):
        self.parent = np.arange(size, dtype=np.int32)

    def find(self, value: int) -> int:
        value = int(value)
        while self.parent[value] != value:
            self.parent[value] = self.parent[self.parent[value]]
            value = int(self.parent[value])
        return value

    def union(self, left: int, right: int) -> None:
        left_root = self.find(left)
        right_root = self.find(right)
        if left_root != right_root:
            self.parent[right_root] = left_root


def connected_components_by_class(class_grid: np.ndarray, valid_mask: np.ndarray) -> np.ndarray:
    """Return dense connected labels and join matching classes across the longitude seam."""
    if class_grid.shape != valid_mask.shape:
        raise ValueError("Class grid and valid mask shapes differ")
    labels = np.zeros(class_grid.shape, dtype=np.int32)
    next_label = 1
    structure = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=np.int8)
    for class_value in np.unique(class_grid[valid_mask]):
        local_labels, local_count = ndimage.label(valid_mask & (class_grid == class_value), structure=structure)
        if local_count:
            occupied = local_labels > 0
            labels[occupied] = local_labels[occupied] + next_label - 1
            next_label += int(local_count)

    disjoint = DisjointSet(next_label)
    seam_rows = np.flatnonzero(
        valid_mask[:, 0]
        & valid_mask[:, -1]
        & (class_grid[:, 0] == class_grid[:, -1])
    )
    for row in seam_rows:
        disjoint.union(int(labels[row, 0]), int(labels[row, -1]))

    occupied_labels = labels[labels > 0]
    roots = np.array([disjoint.find(value) for value in occupied_labels], dtype=np.int32)
    unique_roots, dense_values = np.unique(roots, return_inverse=True)
    del unique_roots
    dense_labels = np.zeros_like(labels)
    dense_labels[labels > 0] = dense_values + 1
    return dense_labels


def dense_relabel(labels: np.ndarray) -> np.ndarray:
    occupied = labels > 0
    unique_labels = np.unique(labels[occupied])
    dense = np.zeros_like(labels, dtype=np.int32)
    if len(unique_labels):
        dense[occupied] = np.searchsorted(unique_labels, labels[occupied]).astype(np.int32) + 1
    return dense


def region_adjacency_counts(labels: np.ndarray, valid_mask: np.ndarray) -> dict[tuple[int, int], int]:
    adjacency: defaultdict[tuple[int, int], int] = defaultdict(int)
    east_labels = np.roll(labels, -1, axis=1)
    east_valid = np.roll(valid_mask, -1, axis=1)
    east_boundary = valid_mask & east_valid & (labels != east_labels)
    for left, right in zip(labels[east_boundary], east_labels[east_boundary]):
        adjacency[tuple(sorted((int(left), int(right))))] += 1

    south_boundary = valid_mask[:-1, :] & valid_mask[1:, :] & (labels[:-1, :] != labels[1:, :])
    for upper, lower in zip(labels[:-1, :][south_boundary], labels[1:, :][south_boundary]):
        adjacency[tuple(sorted((int(upper), int(lower))))] += 1
    return dict(adjacency)


def merge_tiny_adjacent_regions(
    labels: np.ndarray,
    valid_mask: np.ndarray,
    matrix: np.ndarray,
    rows: np.ndarray,
    columns: np.ndarray,
    minimum_cells: int,
) -> tuple[np.ndarray, dict[str, Any]]:
    """Merge small components only into feature-similar, directly adjacent regions."""
    labels = dense_relabel(labels)
    initial_count = int(labels.max())
    if minimum_cells <= 1 or initial_count <= 1:
        return labels, {
            "initial_regions": initial_count,
            "merged_regions": initial_count,
            "remaining_tiny_regions": 0,
            "merge_rounds": 0,
        }

    rounds_completed = 0
    for _ in range(CFG.small_region_merge_rounds):
        cell_labels = labels[rows, columns]
        region_count = int(labels.max())
        sizes = np.bincount(cell_labels, minlength=region_count + 1)
        tiny_labels = np.flatnonzero((sizes > 0) & (sizes < minimum_cells))
        tiny_labels = tiny_labels[tiny_labels > 0]
        if not len(tiny_labels):
            break

        sums = np.zeros((region_count + 1, matrix.shape[1]), dtype=np.float64)
        np.add.at(sums, cell_labels, matrix)
        means = np.divide(
            sums,
            sizes[:, None],
            out=np.zeros_like(sums),
            where=sizes[:, None] > 0,
        )
        adjacency = region_adjacency_counts(labels, valid_mask)
        if not adjacency:
            break

        edge_distances: dict[tuple[int, int], float] = {
            edge: float(np.sqrt(np.mean(np.square(means[edge[0]] - means[edge[1]]))))
            for edge in adjacency
        }
        distance_limit = min(
            float(np.quantile(list(edge_distances.values()), CFG.small_region_merge_distance_quantile)),
            CFG.small_region_merge_max_distance,
        )
        neighbors: defaultdict[int, list[tuple[int, int]]] = defaultdict(list)
        for (left, right), shared_edges in adjacency.items():
            neighbors[left].append((right, shared_edges))
            neighbors[right].append((left, shared_edges))

        disjoint = DisjointSet(region_count + 1)
        accepted_merges = 0
        for region_label in sorted(tiny_labels, key=lambda value: (sizes[value], value)):
            candidates = neighbors.get(int(region_label), [])
            if not candidates:
                continue
            scored = []
            for neighbor_label, shared_edges in candidates:
                edge = tuple(sorted((int(region_label), int(neighbor_label))))
                distance = edge_distances[edge]
                score = distance / (1.0 + 0.10 * math.log1p(shared_edges))
                scored.append((score, distance, -shared_edges, int(neighbor_label)))
            _, distance, _, best_neighbor = min(scored)
            if distance <= distance_limit:
                disjoint.union(int(region_label), best_neighbor)
                accepted_merges += 1

        if accepted_merges == 0:
            break
        roots = np.arange(region_count + 1, dtype=np.int32)
        roots[1:] = [disjoint.find(value) for value in range(1, region_count + 1)]
        previous_count = region_count
        labels = dense_relabel(roots[labels])
        rounds_completed += 1
        if int(labels.max()) >= previous_count:
            break

    final_sizes = np.bincount(labels[rows, columns], minlength=int(labels.max()) + 1)
    remaining_tiny = int(np.sum((final_sizes[1:] > 0) & (final_sizes[1:] < minimum_cells)))
    return labels, {
        "initial_regions": initial_count,
        "merged_regions": int(labels.max()),
        "remaining_tiny_regions": remaining_tiny,
        "merge_rounds": rounds_completed,
    }


def segmentation_prototype_count(scale_name: str, cell_count: int, science_group_count: int) -> int:
    if cell_count < 1:
        raise ValueError("Cannot segment an empty partition")
    if scale_name == "medium_1deg":
        requested = CFG.medium_segmentation_prototypes
        minimum_cells_per_prototype = 50
    else:
        requested = CFG.coarse_segmentation_prototypes
        minimum_cells_per_prototype = 4
    if science_group_count <= 2:
        requested = min(requested, 6)
    if cell_count < 1_000:
        requested = min(requested, 4)
    proposed = max(1, cell_count // minimum_cells_per_prototype)
    return min(cell_count, requested, proposed)


partition_manifest: list[dict[str, Any]] = []
segmentation_models: dict[str, KMeans] = {}
scale_label_rasters: dict[str, np.ndarray] = {}
scale_frames: dict[str, pd.DataFrame] = {}

for scale_name, scale_frame_unsorted in cells_df.groupby("scale", observed=True):
    scale_frame = scale_frame_unsorted.sort_values(["row", "column"]).copy()
    shape = grid_shape(scale_frame)
    scale_frames[str(scale_name)] = scale_frame
    region_labels = np.zeros(shape, dtype=np.int32)
    label_offset = 0

    for cohort_name, partition in scale_frame.groupby("analysis_cohort", observed=True):
        partition = partition.copy()
        selected_features, selected_groups, group_coverages = select_partition_features(
            partition,
            str(scale_name),
            active_schema_df,
            CFG.minimum_group_coverage,
        )
        matrix, imputer_medians = prepare_partition_matrix(partition, selected_features)
        row_indices = partition["row"].to_numpy(dtype=np.int32)
        column_indices = partition["column"].to_numpy(dtype=np.int32)
        smoothing_cells = CFG.medium_smoothing_cells if scale_name == "medium_1deg" else CFG.coarse_smoothing_cells
        segmentation_matrix = smooth_partition_matrix(
            matrix,
            row_indices,
            column_indices,
            shape,
            smoothing_cells,
        )
        prototype_count = segmentation_prototype_count(str(scale_name), len(partition), len(selected_groups))
        prototype_model = KMeans(
            n_clusters=prototype_count,
            n_init=10,
            random_state=CFG.random_seed,
        )
        area_weights = np.clip(
            np.cos(np.deg2rad(partition["latitude"].to_numpy(dtype=np.float64))),
            1e-3,
            None,
        )
        area_weights /= area_weights.mean()
        prototype_labels = prototype_model.fit_predict(
            segmentation_matrix,
            sample_weight=area_weights,
        )
        class_grid = np.full(shape, -1, dtype=np.int16)
        valid_mask = np.zeros(shape, dtype=bool)
        class_grid[row_indices, column_indices] = prototype_labels.astype(np.int16)
        valid_mask[row_indices, column_indices] = True
        minimum_cells = (
            CFG.minimum_medium_region_cells
            if scale_name == "medium_1deg"
            else CFG.minimum_coarse_region_cells
        )
        support_components = connected_components_by_class(np.zeros(shape, dtype=np.int8), valid_mask)
        support_sizes = np.bincount(support_components[valid_mask])
        fragmented_support_cells = int(support_sizes[1:][support_sizes[1:] < minimum_cells].sum())
        fragmented_support_fraction = fragmented_support_cells / len(partition)
        coverage_fragmented = fragmented_support_fraction > CFG.maximum_fragmented_support_fraction
        initial_regions = connected_components_by_class(class_grid, valid_mask)
        local_regions, merge_diagnostics = merge_tiny_adjacent_regions(
            initial_regions,
            valid_mask,
            matrix,
            row_indices,
            column_indices,
            minimum_cells,
        )
        local_region_count = int(local_regions.max())
        region_labels[valid_mask] = local_regions[valid_mask] + label_offset

        partition_id = f"{scale_name}__{cohort_name}"
        segmentation_models[partition_id] = prototype_model
        partition_manifest.append(
            {
                "partition_id": partition_id,
                "scale": str(scale_name),
                "analysis_cohort": str(cohort_name),
                "cell_count": int(len(partition)),
                "selected_features": selected_features,
                "selected_groups": selected_groups,
                "selected_feature_count": len(selected_features),
                "selected_group_count": len(selected_groups),
                "limited_feature_evidence": len(selected_groups) < 2,
                "coverage_fragmented": coverage_fragmented,
                "fragmented_support_fraction": fragmented_support_fraction,
                "group_minimum_finite_fraction": group_coverages,
                "imputer_medians": imputer_medians,
                "segmentation_prototypes": prototype_count,
                "smoothing_cells": smoothing_cells,
                "initial_connected_regions": merge_diagnostics["initial_regions"],
                "connected_regions": local_region_count,
                "remaining_tiny_regions": merge_diagnostics["remaining_tiny_regions"],
                "merge_rounds": merge_diagnostics["merge_rounds"],
            }
        )
        label_offset += local_region_count

    if np.any(region_labels <= 0):
        raise ValueError(f"{scale_name}: segmentation left cells without a connected region")
    scale_label_rasters[str(scale_name)] = region_labels
    print(f"{scale_name}: {label_offset:,} connected regions")

partition_manifest_df = pd.DataFrame(partition_manifest)
display(
    partition_manifest_df[
        [
            "scale",
            "analysis_cohort",
            "cell_count",
            "segmentation_prototypes",
            "initial_connected_regions",
            "connected_regions",
            "remaining_tiny_regions",
            "coverage_fragmented",
            "fragmented_support_fraction",
            "selected_groups",
        ]
    ]
)

## Build region geometry, hierarchy, adjacency, and summaries

Region summaries retain physical-unit means and internal variation, model-space summaries, coverage/quality context, area, perimeter, neighbors, and the dominant spatial fold. Medium regions are linked to the 10° region containing most of their cells; coarse neutron evidence is never copied into the 1° rows.

In [ ]:
def spherical_cell_area_km2(latitude_min: np.ndarray, latitude_max: np.ndarray, longitude_width_deg: float) -> np.ndarray:
    longitude_width_rad = np.deg2rad(longitude_width_deg)
    return (
        CFG.mars_radius_km**2
        * longitude_width_rad
        * (np.sin(np.deg2rad(latitude_max)) - np.sin(np.deg2rad(latitude_min)))
    )


def mode_value(series: pd.Series) -> Any:
    modes = series.mode(dropna=True)
    return modes.iloc[0] if len(modes) else pd.NA


def build_boundary_and_adjacency(
    labels: np.ndarray,
    cell_degrees: float,
    scale_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    maximum_label = int(labels.max())
    perimeter_km = np.zeros(maximum_label + 1, dtype=np.float64)
    boundary_edges = np.zeros(maximum_label + 1, dtype=np.int64)
    adjacency_lengths: defaultdict[tuple[int, int], float] = defaultdict(float)
    north_south_edge_km = CFG.mars_radius_km * np.deg2rad(cell_degrees)

    east = np.roll(labels, -1, axis=1)
    east_difference = labels != east
    east_rows, east_columns = np.nonzero(east_difference)
    for row, column in zip(east_rows, east_columns):
        left = int(labels[row, column])
        right = int(east[row, column])
        perimeter_km[left] += north_south_edge_km
        perimeter_km[right] += north_south_edge_km
        boundary_edges[left] += 1
        boundary_edges[right] += 1
        adjacency_lengths[tuple(sorted((left, right)))] += north_south_edge_km

    south = labels[1:, :]
    north = labels[:-1, :]
    south_difference = north != south
    horizontal_rows, horizontal_columns = np.nonzero(south_difference)
    for row, column in zip(horizontal_rows, horizontal_columns):
        upper = int(north[row, column])
        lower = int(south[row, column])
        boundary_latitude = 90.0 - (row + 1) * cell_degrees
        east_west_edge_km = CFG.mars_radius_km * np.deg2rad(cell_degrees) * max(
            math.cos(math.radians(boundary_latitude)), 0.0
        )
        perimeter_km[upper] += east_west_edge_km
        perimeter_km[lower] += east_west_edge_km
        boundary_edges[upper] += 1
        boundary_edges[lower] += 1
        adjacency_lengths[tuple(sorted((upper, lower)))] += east_west_edge_km

    geometry_df = pd.DataFrame(
        {
            "region_numeric_id": np.arange(1, maximum_label + 1, dtype=np.int32),
            "perimeter_km": perimeter_km[1:],
            "boundary_edge_count": boundary_edges[1:],
        }
    )
    adjacency_df = pd.DataFrame(
        [
            {
                "scale": scale_name,
                "left_region_numeric_id": left,
                "right_region_numeric_id": right,
                "shared_boundary_km": shared_length,
            }
            for (left, right), shared_length in adjacency_lengths.items()
            if left > 0 and right > 0
        ]
    )
    return geometry_df, adjacency_df


def weighted_group_centroids(frame: pd.DataFrame) -> pd.DataFrame:
    work = frame[["region_numeric_id", "latitude", "longitude", "cell_area_km2"]].copy()
    longitude_radians = np.deg2rad(work["longitude"].to_numpy())
    work["weighted_latitude"] = work["latitude"] * work["cell_area_km2"]
    work["weighted_lon_sin"] = np.sin(longitude_radians) * work["cell_area_km2"]
    work["weighted_lon_cos"] = np.cos(longitude_radians) * work["cell_area_km2"]
    sums = work.groupby("region_numeric_id", observed=True)[
        ["cell_area_km2", "weighted_latitude", "weighted_lon_sin", "weighted_lon_cos"]
    ].sum()
    result = pd.DataFrame(index=sums.index)
    result["centroid_latitude"] = sums["weighted_latitude"] / sums["cell_area_km2"]
    result["centroid_longitude"] = np.rad2deg(np.arctan2(sums["weighted_lon_sin"], sums["weighted_lon_cos"]))
    return result.reset_index()


def area_weighted_group_mean(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    values = frame[columns].apply(pd.to_numeric, errors="coerce")
    weights = frame["cell_area_km2"].astype(float)
    weighted_values = values.mul(weights, axis=0)
    valid_weights = values.notna().mul(weights, axis=0)
    numerators = weighted_values.groupby(frame["region_numeric_id"], observed=True).sum(min_count=1)
    denominators = valid_weights.groupby(frame["region_numeric_id"], observed=True).sum(min_count=1)
    return numerators.divide(denominators.where(denominators > 0))


def summarize_regions(
    scale_frame: pd.DataFrame,
    labels: np.ndarray,
    scale_name: str,
    schema: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    frame = scale_frame.copy()
    frame["region_numeric_id"] = labels[
        frame["row"].to_numpy(dtype=np.int32), frame["column"].to_numpy(dtype=np.int32)
    ]
    nominal_degrees = float(frame["nominal_cell_degrees"].iloc[0])
    frame["cell_area_km2"] = spherical_cell_area_km2(
        frame["latitude_min"].to_numpy(dtype=float),
        frame["latitude_max"].to_numpy(dtype=float),
        nominal_degrees,
    )
    prefix = "mars_medium" if scale_name == "medium_1deg" else "mars_coarse"
    frame["region_id"] = [f"{prefix}_region_{value:06d}" for value in frame["region_numeric_id"]]

    scale_schema = schema.loc[(schema["scale"] == scale_name) & schema["active_for_model"]]
    science_features = [feature for feature in scale_schema["feature"] if feature in frame.columns]
    model_columns = [f"model__{feature}" for feature in science_features if f"model__{feature}" in frame.columns]
    quality_features = [
        feature
        for feature in schema.loc[(schema["scale"] == scale_name) & (schema["role"] == "quality_context"), "feature"]
        if feature in frame.columns and pd.api.types.is_numeric_dtype(frame[feature])
    ]

    grouped = frame.groupby("region_numeric_id", observed=True)
    base = grouped.agg(
        region_id=("region_id", "first"),
        scale=("scale", "first"),
        analysis_cohort=("analysis_cohort", "first"),
        cell_count=("cell_id", "size"),
        area_km2=("cell_area_km2", "sum"),
        latitude_min=("latitude_min", "min"),
        latitude_max=("latitude_max", "max"),
        longitude_min=("longitude_min", "min"),
        longitude_max=("longitude_max", "max"),
        dominant_spatial_fold=("spatial_fold", mode_value),
    ).reset_index()
    base = base.merge(weighted_group_centroids(frame), on="region_numeric_id", how="left", validate="one_to_one")

    if science_features:
        raw_mean = area_weighted_group_mean(frame, science_features).add_prefix("raw_mean__")
        raw_std = grouped[science_features].std(ddof=0).add_prefix("raw_std__")
        base = base.merge(raw_mean, on="region_numeric_id", how="left", validate="one_to_one")
        base = base.merge(raw_std, on="region_numeric_id", how="left", validate="one_to_one")
    detection_features = [feature for feature in science_features if feature.startswith("detection_supported__")]
    if detection_features:
        detection_binary = frame[["region_numeric_id", "cell_area_km2", *detection_features]].copy()
        detection_binary[detection_features] = detection_binary[detection_features].where(
            detection_binary[detection_features].isna(),
            detection_binary[detection_features] > 0,
        )
        positive_fraction = area_weighted_group_mean(detection_binary, detection_features)
        positive_fraction = positive_fraction.add_prefix("raw_positive_fraction__")
        base = base.merge(positive_fraction, on="region_numeric_id", how="left", validate="one_to_one")
    if model_columns:
        model_mean = area_weighted_group_mean(frame, model_columns).rename(columns=lambda column: f"model_mean__{column.removeprefix('model__')}")
        model_std = grouped[model_columns].std(ddof=0).rename(columns=lambda column: f"model_std__{column.removeprefix('model__')}")
        base = base.merge(model_mean, on="region_numeric_id", how="left", validate="one_to_one")
        base = base.merge(model_std, on="region_numeric_id", how="left", validate="one_to_one")
    if quality_features:
        quality_mean = area_weighted_group_mean(frame, quality_features).add_prefix("quality_mean__")
        base = base.merge(quality_mean, on="region_numeric_id", how="left", validate="one_to_one")

    geometry_df, adjacency_df = build_boundary_and_adjacency(labels, nominal_degrees, scale_name)
    base = base.merge(geometry_df, on="region_numeric_id", how="left", validate="one_to_one")
    base["compactness"] = np.divide(
        4.0 * np.pi * base["area_km2"],
        np.square(base["perimeter_km"]),
        out=np.zeros(len(base), dtype=float),
        where=base["perimeter_km"].to_numpy() > 0,
    )
    minimum_cells = CFG.minimum_medium_region_cells if scale_name == "medium_1deg" else CFG.minimum_coarse_region_cells
    base["is_tiny_region"] = base["cell_count"] < minimum_cells
    return frame, base, adjacency_df


region_cell_frames: list[pd.DataFrame] = []
region_frames: list[pd.DataFrame] = []
adjacency_frames: list[pd.DataFrame] = []
for scale_name, scale_frame in scale_frames.items():
    region_cells, regions, adjacency = summarize_regions(
        scale_frame,
        scale_label_rasters[scale_name],
        scale_name,
        feature_schema_df,
    )
    region_cell_frames.append(region_cells)
    region_frames.append(regions)
    adjacency_frames.append(adjacency)

region_cells_df = pd.concat(region_cell_frames, ignore_index=True, sort=False)
regions_df = pd.concat(region_frames, ignore_index=True, sort=False)
adjacency_df = pd.concat(adjacency_frames, ignore_index=True, sort=False)

region_lookup = regions_df.set_index(["scale", "region_numeric_id"])["region_id"]
adjacency_df["left_region_id"] = [region_lookup.loc[(scale, value)] for scale, value in zip(adjacency_df["scale"], adjacency_df["left_region_numeric_id"])]
adjacency_df["right_region_id"] = [region_lookup.loc[(scale, value)] for scale, value in zip(adjacency_df["scale"], adjacency_df["right_region_numeric_id"])]

# Link each medium region to the coarse region covering most of its cells.
coarse_cell_to_region = region_cells_df.loc[region_cells_df["scale"] == "coarse_10deg"].set_index("cell_id")["region_id"]
medium_links = region_cells_df.loc[region_cells_df["scale"] == "medium_1deg", ["region_id", "parent_coarse_id"]].copy()
medium_links["parent_region_id"] = medium_links["parent_coarse_id"].map(coarse_cell_to_region)
overlap = medium_links.groupby(["region_id", "parent_region_id"], observed=True).size().rename("overlap_cells").reset_index()
overlap["region_cells"] = overlap.groupby("region_id", observed=True)["overlap_cells"].transform("sum")
overlap["parent_overlap_fraction"] = overlap["overlap_cells"] / overlap["region_cells"]
best_parent = overlap.sort_values(["region_id", "overlap_cells"], ascending=[True, False]).drop_duplicates("region_id")
regions_df = regions_df.merge(
    best_parent[["region_id", "parent_region_id", "parent_overlap_fraction"]],
    on="region_id",
    how="left",
    validate="one_to_one",
)

region_summary = regions_df.groupby(["scale", "analysis_cohort"], observed=True).agg(
    region_count=("region_id", "size"),
    median_cells=("cell_count", "median"),
    tiny_region_percent=("is_tiny_region", lambda values: 100.0 * values.mean()),
    median_area_km2=("area_km2", "median"),
).reset_index()
tiny_coverage_summary = (
    regions_df.assign(
        tiny_cells=np.where(regions_df["is_tiny_region"], regions_df["cell_count"], 0),
        tiny_area_km2=np.where(regions_df["is_tiny_region"], regions_df["area_km2"], 0.0),
    )
    .groupby(["scale", "analysis_cohort"], observed=True)
    .agg(
        total_cells=("cell_count", "sum"),
        tiny_cells=("tiny_cells", "sum"),
        total_area_km2=("area_km2", "sum"),
        tiny_area_km2=("tiny_area_km2", "sum"),
    )
    .reset_index()
)
tiny_coverage_summary["tiny_cell_percent"] = 100.0 * tiny_coverage_summary["tiny_cells"] / tiny_coverage_summary["total_cells"]
tiny_coverage_summary["tiny_area_percent"] = 100.0 * tiny_coverage_summary["tiny_area_km2"] / tiny_coverage_summary["total_area_km2"]
region_summary = region_summary.merge(
    tiny_coverage_summary[["scale", "analysis_cohort", "tiny_cell_percent", "tiny_area_percent"]],
    on=["scale", "analysis_cohort"],
    how="left",
    validate="one_to_one",
)
display(region_summary)

## Discover neutral region types in two rounds

The first round uses only broad, comparatively robust properties: terrain, reflectance, thermal behavior where available, and coarse hydrogen-sensitive measurements at 10°. The second round may split a broad type using composition, spectral shape, or surface-dust measurements, but only when enough regions have those measurements and the split is not predictable from acquisition quality.

This hierarchy prevents patchy spectroscopy from defining the global types while still allowing richer measurements to reveal scientifically interesting subtypes. Regions below the minimum size are excluded from preprocessing, model fitting, validation, artifact testing, and anomaly thresholds so that many one-cell fragments cannot bias the result. After fitting, those fragments receive a transparent post-hoc assignment from the chosen model (or nearest HDBSCAN type) and are flagged in the exports. Readiness is judged by the fraction of cells/area affected, not by the misleading fraction of region records.

At each round, K-means, Gaussian mixtures, and HDBSCAN are compared using geographic-fold stability, silhouette separation, assigned coverage, and a coverage-artifact test. Broad types use the best safe candidate. A subtype is applied only if it has at least 40 substantive training regions, at least 10 regions per resulting cluster, spatial ARI ≥ 0.60, silhouette ≥ 0.20, assigned fraction ≥ 0.60, and no artifact warning. Otherwise the broad type is kept and the rejection reason is exported. K-means confidence is a real nearest-versus-second-nearest distance margin, not a constant value.

API references: [scikit-learn HDBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html), [silhouette score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).


In [ ]:
CORE_GROUPS_BY_SCALE = {
    "medium_1deg": {"terrain", "reflectance", "thermophysical"},
    "coarse_10deg": {"terrain", "reflectance", "thermophysical", "hydrogen_sensitive"},
}
SUBTYPE_GROUPS = {"composition", "spectral_contrast", "spectral_shape", "surface_dust"}

QUALITY_COLUMNS_BY_GROUP = {
    "terrain": ["quality_mean__mola_observed_fraction"],
    "reflectance": ["quality_mean__albedo_coverage"],
    "thermophysical": ["quality_mean__thermal_inertia_coverage"],
    "composition": [
        "quality_mean__mineral_fit_rms_error",
        "quality_mean__mineral_source_pixel_fraction",
        "quality_mean__mineral_valid_band_fraction",
        "quality_mean__mineral_quality_valid",
    ],
    "surface_dust": [
        "quality_mean__mineral_source_pixel_fraction",
        "quality_mean__mineral_valid_band_fraction",
        "quality_mean__mineral_quality_valid",
    ],
    "spectral_contrast": [
        "quality_mean__emissivity_valid_band_fraction",
        "quality_mean__emissivity_source_pixel_fraction",
        "quality_mean__emissivity_shape_reliable",
    ],
    "spectral_shape": [
        "quality_mean__emissivity_valid_band_fraction",
        "quality_mean__emissivity_source_pixel_fraction",
        "quality_mean__emissivity_shape_reliable",
    ],
    "hydrogen_sensitive": ["quality_mean__missing_neutron"],
}


def safe_silhouette(matrix: np.ndarray, labels: np.ndarray) -> float:
    labels = np.asarray(labels)
    usable = labels >= 0
    unique_labels = np.unique(labels[usable])
    if usable.sum() < 3 or len(unique_labels) < 2 or len(unique_labels) >= usable.sum():
        return float("nan")
    sample_size = min(CFG.maximum_silhouette_rows, int(usable.sum()))
    return float(
        silhouette_score(
            matrix[usable],
            labels[usable],
            sample_size=sample_size if sample_size < usable.sum() else None,
            random_state=CFG.random_seed,
        )
    )


def stability_against_spatial_folds(
    matrix: np.ndarray,
    folds: np.ndarray,
    full_labels: np.ndarray,
    fit_predict: Callable[[np.ndarray, np.ndarray], tuple[np.ndarray, np.ndarray]],
    compare_on_training: bool = False,
) -> float:
    scores: list[float] = []
    for fold in sorted(np.unique(folds)):
        training = folds != fold
        testing = ~training
        if training.sum() < 5 or testing.sum() < 2:
            continue
        train_labels, test_labels = fit_predict(matrix[training], matrix[testing])
        if compare_on_training:
            reference = full_labels[training]
            comparison = train_labels
        else:
            reference = full_labels[testing]
            comparison = test_labels
        if len(np.unique(reference)) < 2 or len(np.unique(comparison)) < 2:
            continue
        scores.append(float(adjusted_rand_score(reference, comparison)))
    return float(np.mean(scores)) if scores else float("nan")


def kmeans_margin_confidence(model: KMeans, matrix: np.ndarray) -> np.ndarray:
    distances = model.transform(matrix)
    if distances.shape[1] < 2:
        return np.ones(len(matrix), dtype=np.float32)
    nearest_two = np.partition(distances, kth=1, axis=1)[:, :2]
    nearest_two.sort(axis=1)
    margin = (nearest_two[:, 1] - nearest_two[:, 0]) / np.maximum(nearest_two[:, 1], 1e-12)
    return np.clip(margin, 0.0, 1.0).astype(np.float32)


def choose_kmeans(matrix: np.ndarray, maximum_cluster_count: int) -> tuple[KMeans, np.ndarray, int, float]:
    maximum_k = min(maximum_cluster_count, len(matrix) - 1)
    candidates: list[tuple[float, int, KMeans, np.ndarray]] = []
    for cluster_count in range(2, maximum_k + 1):
        model = KMeans(n_clusters=cluster_count, n_init=20, random_state=CFG.random_seed)
        labels = model.fit_predict(matrix)
        candidates.append((safe_silhouette(matrix, labels), cluster_count, model, labels))
    valid = [candidate for candidate in candidates if np.isfinite(candidate[0])]
    if not valid:
        raise ValueError("K-means did not produce a valid multi-cluster partition")
    silhouette, cluster_count, model, labels = max(valid, key=lambda item: item[0])
    return model, labels, cluster_count, float(silhouette)


def choose_gmm(matrix: np.ndarray, maximum_cluster_count: int) -> tuple[GaussianMixture, np.ndarray, int, float, float]:
    maximum_k = min(maximum_cluster_count, len(matrix) - 1)
    candidates: list[tuple[float, int, GaussianMixture, np.ndarray]] = []
    for component_count in range(2, maximum_k + 1):
        model = GaussianMixture(
            n_components=component_count,
            covariance_type="diag",
            reg_covar=1e-5,
            n_init=3,
            random_state=CFG.random_seed,
        )
        labels = model.fit_predict(matrix)
        candidates.append((float(model.bic(matrix)), component_count, model, labels))
    bic, component_count, model, labels = min(candidates, key=lambda item: item[0])
    return model, labels, component_count, safe_silhouette(matrix, labels), bic


def clustering_selection_score(stability: float, silhouette: float, assigned_fraction: float) -> float:
    stability_term = np.clip(stability if np.isfinite(stability) else -1.0, -1.0, 1.0)
    silhouette_term = np.clip(silhouette if np.isfinite(silhouette) else -1.0, -1.0, 1.0)
    return float(0.50 * ((stability_term + 1.0) / 2.0) + 0.35 * ((silhouette_term + 1.0) / 2.0) + 0.15 * assigned_fraction)


def compare_partition_clusterers(
    matrix: np.ndarray,
    folds: np.ndarray,
    partition_id: str,
    maximum_cluster_count: int,
) -> tuple[pd.DataFrame, dict[str, dict[str, Any]]]:
    candidates: dict[str, dict[str, Any]] = {}

    kmeans_model, kmeans_labels, kmeans_k, kmeans_silhouette = choose_kmeans(matrix, maximum_cluster_count)

    def refit_kmeans(train: np.ndarray, test: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        model = KMeans(n_clusters=kmeans_k, n_init=20, random_state=CFG.random_seed)
        return model.fit_predict(train), model.predict(test)

    kmeans_stability = stability_against_spatial_folds(matrix, folds, kmeans_labels, refit_kmeans)
    candidates["kmeans"] = {
        "model": kmeans_model,
        "labels": kmeans_labels,
        "confidence": kmeans_margin_confidence(kmeans_model, matrix),
        "cluster_count": kmeans_k,
        "noise_fraction": 0.0,
        "silhouette": kmeans_silhouette,
        "stability": kmeans_stability,
        "criterion": "best silhouette across candidate k",
    }

    gmm_model, gmm_labels, gmm_k, gmm_silhouette, gmm_bic = choose_gmm(matrix, maximum_cluster_count)

    def refit_gmm(train: np.ndarray, test: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        model = GaussianMixture(
            n_components=gmm_k,
            covariance_type="diag",
            reg_covar=1e-5,
            n_init=3,
            random_state=CFG.random_seed,
        )
        return model.fit_predict(train), model.predict(test)

    gmm_stability = stability_against_spatial_folds(matrix, folds, gmm_labels, refit_gmm)
    candidates["gaussian_mixture"] = {
        "model": gmm_model,
        "labels": gmm_labels,
        "confidence": gmm_model.predict_proba(matrix).max(axis=1).astype(np.float32),
        "cluster_count": gmm_k,
        "noise_fraction": 0.0,
        "silhouette": gmm_silhouette,
        "stability": gmm_stability,
        "criterion": f"lowest BIC ({gmm_bic:.1f})",
    }

    if len(matrix) <= CFG.maximum_hdbscan_regions:
        minimum_cluster_size = max(5, int(round(0.03 * len(matrix))))
        minimum_samples = max(3, int(round(math.sqrt(minimum_cluster_size))))
        hdbscan_model = HDBSCAN(
            min_cluster_size=minimum_cluster_size,
            min_samples=minimum_samples,
            cluster_selection_method="eom",
            allow_single_cluster=False,
            n_jobs=-1,
        )
        hdbscan_labels = hdbscan_model.fit_predict(matrix)
        hdbscan_cluster_count = len(set(hdbscan_labels) - {-1})

        def refit_hdbscan(train: np.ndarray, test: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
            del test
            model = HDBSCAN(
                min_cluster_size=max(5, int(round(0.03 * len(train)))),
                min_samples=minimum_samples,
                cluster_selection_method="eom",
                allow_single_cluster=False,
                n_jobs=-1,
            )
            train_labels = model.fit_predict(train)
            return train_labels, np.empty(0, dtype=np.int32)

        hdbscan_stability = stability_against_spatial_folds(
            matrix,
            folds,
            hdbscan_labels,
            refit_hdbscan,
            compare_on_training=True,
        )
        candidates["hdbscan"] = {
            "model": hdbscan_model,
            "labels": hdbscan_labels,
            "confidence": np.asarray(hdbscan_model.probabilities_, dtype=np.float32),
            "cluster_count": hdbscan_cluster_count,
            "noise_fraction": float(np.mean(hdbscan_labels < 0)),
            "silhouette": safe_silhouette(matrix, hdbscan_labels),
            "stability": hdbscan_stability,
            "criterion": f"min_cluster_size={minimum_cluster_size}, min_samples={minimum_samples}",
        }

    rows: list[dict[str, Any]] = []
    for algorithm, result in candidates.items():
        valid_partition = result["cluster_count"] >= 2
        assigned_fraction = 1.0 - result["noise_fraction"]
        selection_score = (
            clustering_selection_score(result["stability"], result["silhouette"], assigned_fraction)
            if valid_partition
            else -np.inf
        )
        result["selection_score"] = selection_score
        rows.append(
            {
                "partition_id": partition_id,
                "algorithm": algorithm,
                "cluster_count": result["cluster_count"],
                "noise_fraction": result["noise_fraction"],
                "assigned_fraction": assigned_fraction,
                "minimum_cluster_size": int(
                    np.min(np.bincount(result["labels"][result["labels"] >= 0]))
                    if np.any(result["labels"] >= 0)
                    else 0
                ),
                "silhouette": result["silhouette"],
                "spatial_stability_ari": result["stability"],
                "selection_score": selection_score,
                "criterion": result["criterion"],
            }
        )
    return pd.DataFrame(rows), candidates


def artifact_feature_columns(frame: pd.DataFrame, science_groups: set[str]) -> list[str]:
    requested = {
        column
        for group in science_groups
        for column in QUALITY_COLUMNS_BY_GROUP.get(group, [])
    }
    return [
        column
        for column in sorted(requested)
        if column in frame.columns
        and frame[column].notna().any()
        and frame[column].dropna().nunique() > 1
    ]


def cluster_artifact_score(
    partition: pd.DataFrame,
    labels: np.ndarray,
    science_groups: set[str],
) -> tuple[float, float, bool, bool]:
    labels = np.asarray(labels, dtype=np.int32)
    usable = labels >= 0
    if "model_training_eligible" in partition.columns:
        usable &= partition["model_training_eligible"].to_numpy(dtype=bool)
    columns = artifact_feature_columns(partition, science_groups)
    if usable.sum() < 20 or len(np.unique(labels[usable])) < 2 or len(columns) < 1:
        return float("nan"), float("nan"), False, False

    values = partition[columns].to_numpy(dtype=np.float64)[usable]
    usable_labels = labels[usable]
    folds = partition["dominant_spatial_fold"].to_numpy(dtype=np.int16)[usable]
    fold_scores: list[float] = []
    for fold in sorted(np.unique(folds)):
        training = folds != fold
        testing = ~training
        if training.sum() < 10 or testing.sum() < 3 or len(np.unique(usable_labels[training])) < 2:
            continue
        imputer = SimpleImputer(strategy="median")
        train_values = imputer.fit_transform(values[training])
        test_values = imputer.transform(values[testing])
        classifier = RandomForestClassifier(
            n_estimators=150,
            class_weight="balanced_subsample",
            min_samples_leaf=max(2, int(training.sum() // 500)),
            n_jobs=-1,
            random_state=CFG.random_seed + int(fold),
        )
        classifier.fit(train_values, usable_labels[training])
        predictions = classifier.predict(test_values)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=UserWarning)
            fold_scores.append(float(balanced_accuracy_score(usable_labels[testing], predictions)))
    if not fold_scores:
        return float("nan"), float("nan"), False, False
    score = float(np.mean(fold_scores))
    chance = 1.0 / len(np.unique(usable_labels))
    warning = score >= max(CFG.artifact_warning_balanced_accuracy, chance + 0.25)
    return score, chance, warning, True


def select_candidate_with_artifact_control(
    comparison: pd.DataFrame,
    candidates: dict[str, dict[str, Any]],
    partition: pd.DataFrame,
    science_groups: set[str],
    stage: str,
) -> tuple[pd.DataFrame, str, dict[str, Any], bool]:
    comparison = comparison.copy()
    diagnostics: dict[str, tuple[float, float, bool, bool]] = {}
    for algorithm, result in candidates.items():
        diagnostics[algorithm] = cluster_artifact_score(partition, result["labels"], science_groups)
    comparison["coverage_balanced_accuracy"] = comparison["algorithm"].map(
        {algorithm: values[0] for algorithm, values in diagnostics.items()}
    )
    comparison["chance_balanced_accuracy"] = comparison["algorithm"].map(
        {algorithm: values[1] for algorithm, values in diagnostics.items()}
    )
    comparison["artifact_warning"] = comparison["algorithm"].map(
        {algorithm: values[2] for algorithm, values in diagnostics.items()}
    ).astype(bool)
    comparison["artifact_diagnostic_available"] = comparison["algorithm"].map(
        {algorithm: values[3] for algorithm, values in diagnostics.items()}
    ).astype(bool)
    quality_pass: dict[str, bool] = {}
    quality_reason: dict[str, str] = {}
    for algorithm, result in candidates.items():
        reasons: list[str] = []
        if stage == "subtype":
            labels = np.asarray(result["labels"], dtype=np.int32)
            assigned = labels >= 0
            cluster_sizes = np.bincount(labels[assigned]) if assigned.any() else np.array([], dtype=int)
            if len(labels) < CFG.minimum_subtype_regions:
                reasons.append(f"training_regions<{CFG.minimum_subtype_regions}")
            if len(cluster_sizes) < 2 or (len(cluster_sizes) and cluster_sizes.min() < CFG.minimum_subtype_regions_per_cluster):
                reasons.append(f"cluster_size<{CFG.minimum_subtype_regions_per_cluster}")
            if not np.isfinite(result["stability"]) or result["stability"] < CFG.minimum_subtype_stability_ari:
                reasons.append(f"stability<{CFG.minimum_subtype_stability_ari:.2f}")
            if not np.isfinite(result["silhouette"]) or result["silhouette"] < CFG.minimum_subtype_silhouette:
                reasons.append(f"silhouette<{CFG.minimum_subtype_silhouette:.2f}")
            if (1.0 - result["noise_fraction"]) < CFG.minimum_subtype_assigned_fraction:
                reasons.append(f"assigned_fraction<{CFG.minimum_subtype_assigned_fraction:.2f}")
        quality_pass[algorithm] = not reasons
        quality_reason[algorithm] = "; ".join(reasons) if reasons else "passed"
    comparison["quality_gate_pass"] = comparison["algorithm"].map(quality_pass).astype(bool)
    comparison["quality_gate_reason"] = comparison["algorithm"].map(quality_reason)

    eligible = comparison.loc[
        comparison["artifact_diagnostic_available"]
        & ~comparison["artifact_warning"]
        & comparison["quality_gate_pass"]
        & np.isfinite(comparison["selection_score"])
    ]
    all_candidates_rejected = eligible.empty
    if not eligible.empty:
        selected_row = eligible.sort_values("selection_score", ascending=False).iloc[0]
    else:
        fallback = comparison.loc[np.isfinite(comparison["selection_score"])].copy()
        fallback["artifact_sort"] = fallback["coverage_balanced_accuracy"].fillna(1.0)
        selected_row = fallback.sort_values(["artifact_sort", "selection_score"], ascending=[True, False]).iloc[0]
    selected_algorithm = str(selected_row["algorithm"])
    comparison["best_candidate"] = comparison["algorithm"].eq(selected_algorithm)
    comparison["applied_to_output"] = False
    return comparison, selected_algorithm, candidates[selected_algorithm], all_candidates_rejected


def residualize_quality_effects(
    partition: pd.DataFrame,
    matrix: np.ndarray,
    science_groups: set[str],
    fit_mask: np.ndarray,
) -> tuple[np.ndarray, dict[str, Any] | None, float]:
    """Fit acquisition-quality correction on substantive regions, then transform every region."""
    fit_mask = np.asarray(fit_mask, dtype=bool)
    if fit_mask.shape != (len(partition),) or fit_mask.sum() < CFG.minimum_partition_regions:
        raise ValueError("Quality-correction fit mask is invalid or too small")
    quality_columns = artifact_feature_columns(partition.loc[fit_mask], science_groups)
    if len(quality_columns) < 1 or fit_mask.sum() < 20:
        return matrix.astype(np.float32), None, 0.0

    quality_imputer = SimpleImputer(strategy="median")
    raw_quality = partition[quality_columns].to_numpy(dtype=np.float64)
    quality_imputer.fit(raw_quality[fit_mask])
    quality_values = quality_imputer.transform(raw_quality)
    variable = np.nanstd(quality_values[fit_mask], axis=0) > 1e-8
    if variable.sum() < 1:
        return matrix.astype(np.float32), None, 0.0
    quality_columns = [column for column, keep in zip(quality_columns, variable) if keep]
    quality_values = quality_values[:, variable]
    quality_scaler = StandardScaler()
    quality_scaler.fit(quality_values[fit_mask])
    quality_scaled = quality_scaler.transform(quality_values)
    area_weights = partition["area_km2"].to_numpy(dtype=np.float64)
    area_weights = np.clip(area_weights, 1e-9, None)
    fit_weights = area_weights[fit_mask]
    fit_weights /= fit_weights.mean()

    ridge = Ridge(alpha=CFG.quality_residualization_alpha)
    ridge.fit(quality_scaled[fit_mask], matrix[fit_mask], sample_weight=fit_weights)
    predicted = ridge.predict(quality_scaled)
    predicted_center = np.average(predicted[fit_mask], axis=0, weights=fit_weights)
    corrected = matrix - (predicted - predicted_center)

    target_center = np.average(matrix[fit_mask], axis=0, weights=fit_weights)
    total_variance = np.average(np.square(matrix[fit_mask] - target_center), axis=0, weights=fit_weights)
    residual_variance = np.average(np.square(matrix[fit_mask] - predicted[fit_mask]), axis=0, weights=fit_weights)
    explained = np.divide(
        total_variance - residual_variance,
        total_variance,
        out=np.zeros_like(total_variance),
        where=total_variance > 1e-12,
    )
    explained_fraction = float(np.clip(np.nanmean(explained), 0.0, 1.0))

    feature_medians = np.median(corrected[fit_mask], axis=0)
    q25, q75 = np.quantile(corrected[fit_mask], [0.25, 0.75], axis=0)
    feature_scales = q75 - q25
    feature_scales = np.where(feature_scales > 1e-8, feature_scales, 1.0)
    corrected = np.clip((corrected - feature_medians) / feature_scales, -8.0, 8.0).astype(np.float32)
    correction = {
        "quality_columns": quality_columns,
        "imputer": quality_imputer,
        "scaler": quality_scaler,
        "ridge": ridge,
        "prediction_center": predicted_center,
        "feature_medians": feature_medians,
        "feature_scales": feature_scales,
        "fit_region_count": int(fit_mask.sum()),
    }
    return corrected, correction, explained_fraction


def partition_region_matrix(
    partition: pd.DataFrame,
    manifest_row: pd.Series,
    allowed_groups: set[str],
    fit_mask: np.ndarray,
) -> tuple[np.ndarray, list[str], dict[str, str], set[str], dict[str, Any] | None, float]:
    scale_name = str(manifest_row["scale"])
    schema_rows = active_schema_df.loc[active_schema_df["scale"] == scale_name, ["feature", "group"]]
    feature_to_group = schema_rows.set_index("feature")["group"].astype(str).to_dict()
    features = [
        feature
        for feature in list(manifest_row["selected_features"])
        if feature_to_group.get(feature) in allowed_groups
    ]
    columns = [f"model_mean__{feature}" for feature in features]
    fit_mask = np.asarray(fit_mask, dtype=bool)
    if fit_mask.shape != (len(partition),) or fit_mask.sum() < CFG.minimum_partition_regions:
        raise ValueError(f"{manifest_row['partition_id']}: too few substantive regions for model fitting")
    available = [
        column
        for column in columns
        if column in partition.columns and partition.loc[fit_mask, column].notna().mean() >= CFG.minimum_group_coverage
    ]
    if len(available) < 2:
        raise ValueError(f"{manifest_row['partition_id']}: fewer than two usable features for groups {sorted(allowed_groups)}")
    values = partition[available].to_numpy(dtype=np.float64)
    measurement_imputer = SimpleImputer(strategy="median")
    measurement_imputer.fit(values[fit_mask])
    matrix = measurement_imputer.transform(values)
    matrix = np.clip(matrix, -8.0, 8.0).astype(np.float32)
    column_to_group = {
        column: feature_to_group[column.removeprefix("model_mean__")]
        for column in available
    }
    science_groups = set(column_to_group.values())
    matrix, quality_correction, quality_explained_fraction = residualize_quality_effects(
        partition, matrix, science_groups, fit_mask
    )
    return matrix, available, column_to_group, science_groups, quality_correction, quality_explained_fraction


def assign_excluded_regions(
    algorithm: str,
    candidate: dict[str, Any],
    full_matrix: np.ndarray,
    fit_mask: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Keep fitted labels intact and assign excluded tiny fragments after model fitting."""
    fit_mask = np.asarray(fit_mask, dtype=bool)
    training_labels = np.asarray(candidate["labels"], dtype=np.int32)
    training_confidence = np.asarray(candidate["confidence"], dtype=np.float32)
    if len(training_labels) != int(fit_mask.sum()):
        raise ValueError("Candidate labels do not match the model-training region count")

    labels = np.full(len(full_matrix), -1, dtype=np.int32)
    confidence = np.zeros(len(full_matrix), dtype=np.float32)
    labels[fit_mask] = training_labels
    confidence[fit_mask] = training_confidence
    excluded = ~fit_mask
    if not excluded.any():
        return labels, confidence

    model = candidate["model"]
    if algorithm == "kmeans":
        labels[excluded] = model.predict(full_matrix[excluded]).astype(np.int32)
        confidence[excluded] = kmeans_margin_confidence(model, full_matrix[excluded])
        return labels, confidence
    if algorithm == "gaussian_mixture":
        labels[excluded] = model.predict(full_matrix[excluded]).astype(np.int32)
        confidence[excluded] = model.predict_proba(full_matrix[excluded]).max(axis=1).astype(np.float32)
        return labels, confidence

    valid_training = training_labels >= 0
    cluster_labels = np.unique(training_labels[valid_training])
    if len(cluster_labels) < 1:
        return labels, confidence
    training_matrix = full_matrix[fit_mask]
    centroids = np.vstack([training_matrix[training_labels == label].mean(axis=0) for label in cluster_labels])
    distances = np.linalg.norm(full_matrix[excluded, None, :] - centroids[None, :, :], axis=2)
    nearest = np.argmin(distances, axis=1)
    labels[excluded] = cluster_labels[nearest].astype(np.int32)
    if len(cluster_labels) > 1:
        nearest_two = np.partition(distances, kth=1, axis=1)[:, :2]
        nearest_two.sort(axis=1)
        confidence[excluded] = np.clip(
            (nearest_two[:, 1] - nearest_two[:, 0]) / np.maximum(nearest_two[:, 1], 1e-12), 0.0, 1.0
        ).astype(np.float32)
    else:
        within = np.linalg.norm(training_matrix[valid_training] - centroids[0], axis=1)
        distance_scale = max(float(np.median(within)), 1e-6)
        confidence[excluded] = np.exp(-distances[:, 0] / distance_scale).astype(np.float32)
    return labels, confidence


comparison_frames: list[pd.DataFrame] = []
selected_models: dict[str, Any] = {}
quality_correction_models: dict[str, dict[str, Any]] = {}
quality_correction_rows: list[dict[str, Any]] = []
partition_cluster_context: dict[str, dict[str, Any]] = {}
regions_df["partition_id"] = regions_df["scale"] + "__" + regions_df["analysis_cohort"].astype(str)
limited_evidence_lookup = partition_manifest_df.set_index("partition_id")["limited_feature_evidence"].to_dict()
fragmented_lookup = partition_manifest_df.set_index("partition_id")["coverage_fragmented"].to_dict()
regions_df["limited_feature_evidence"] = regions_df["partition_id"].map(limited_evidence_lookup).eq(True)
regions_df["coverage_fragmented"] = regions_df["partition_id"].map(fragmented_lookup).eq(True)
regions_df["model_training_eligible"] = ~regions_df["is_tiny_region"] & ~regions_df["coverage_fragmented"]
regions_df["cluster_assignment_role"] = "model_training"
regions_df.loc[regions_df["is_tiny_region"], "cluster_assignment_role"] = "posthoc_tiny"
regions_df.loc[regions_df["coverage_fragmented"], "cluster_assignment_role"] = "descriptive_only"
regions_df["broad_local_label"] = -999
regions_df["subtype_local_label"] = -1
regions_df["cluster_local_label"] = -999
regions_df["broad_cluster_confidence"] = np.nan
regions_df["subtype_cluster_confidence"] = np.nan
regions_df["cluster_confidence"] = np.nan
regions_df["selected_algorithm"] = pd.NA
regions_df["broad_type"] = pd.NA
regions_df["neutral_type"] = pd.NA
regions_df["rare_broad_type"] = False

for manifest_row in partition_manifest_df.itertuples(index=False):
    manifest = pd.Series(manifest_row._asdict())
    partition_id = str(manifest_row.partition_id)
    partition_mask = regions_df["partition_id"] == partition_id
    partition = regions_df.loc[partition_mask].copy()
    scale_name = str(manifest_row.scale)
    broad_fit_mask = partition["model_training_eligible"].to_numpy(dtype=bool)

    if bool(manifest_row.coverage_fragmented):
        unresolved_name = f"mars_{partition_id}_coverage_fragmented_unresolved"
        regions_df.loc[partition.index, "broad_local_label"] = -1
        regions_df.loc[partition.index, "cluster_local_label"] = -1
        regions_df.loc[partition.index, "selected_algorithm"] = "not_clustered_fragmented_coverage"
        regions_df.loc[partition.index, "broad_type"] = unresolved_name
        regions_df.loc[partition.index, "neutral_type"] = unresolved_name
        continue

    if broad_fit_mask.sum() < CFG.minimum_partition_regions:
        single_name = f"mars_{partition_id}_broad_00"
        regions_df.loc[partition.index, "broad_local_label"] = 0
        regions_df.loc[partition.index, "cluster_local_label"] = 0
        regions_df.loc[partition.index, "broad_cluster_confidence"] = 1.0
        regions_df.loc[partition.index, "cluster_confidence"] = 1.0
        regions_df.loc[partition.index, "selected_algorithm"] = "single_type_too_few_substantive_regions"
        regions_df.loc[partition.index, "cluster_assignment_role"] = "descriptive_only"
        regions_df.loc[partition.index, "broad_type"] = single_name
        regions_df.loc[partition.index, "neutral_type"] = single_name
        continue

    try:
        (
            broad_matrix,
            broad_columns,
            broad_column_to_group,
            broad_groups,
            broad_quality_correction,
            broad_quality_explained,
        ) = partition_region_matrix(partition, manifest, CORE_GROUPS_BY_SCALE[scale_name], broad_fit_mask)
    except ValueError:
        unresolved_name = f"mars_{partition_id}_insufficient_core_features"
        regions_df.loc[partition.index, "broad_local_label"] = -1
        regions_df.loc[partition.index, "cluster_local_label"] = -1
        regions_df.loc[partition.index, "selected_algorithm"] = "not_clustered_insufficient_core_features"
        regions_df.loc[partition.index, "cluster_assignment_role"] = "descriptive_only"
        regions_df.loc[partition.index, "broad_type"] = unresolved_name
        regions_df.loc[partition.index, "neutral_type"] = unresolved_name
        regions_df.loc[partition.index, "limited_feature_evidence"] = True
        continue

    training_partition = partition.loc[broad_fit_mask].copy()
    folds = training_partition["dominant_spatial_fold"].to_numpy(dtype=np.int16)
    broad_comparison, broad_candidates = compare_partition_clusterers(
        broad_matrix[broad_fit_mask],
        folds,
        partition_id,
        CFG.maximum_cluster_count,
    )
    broad_comparison, broad_algorithm, broad_selected, broad_all_rejected = select_candidate_with_artifact_control(
        broad_comparison,
        broad_candidates,
        training_partition,
        broad_groups,
        stage="broad",
    )
    broad_comparison["stage"] = "broad"
    broad_comparison["parent_broad_type"] = pd.NA
    broad_comparison["all_candidates_rejected"] = broad_all_rejected
    broad_comparison.loc[broad_comparison["best_candidate"], "applied_to_output"] = True
    broad_comparison["model_training_region_count"] = int(broad_fit_mask.sum())
    broad_comparison["posthoc_assigned_region_count"] = int((~broad_fit_mask).sum())
    comparison_frames.append(broad_comparison)

    selected_models[f"{partition_id}__broad"] = broad_selected["model"]
    if broad_quality_correction is not None:
        quality_correction_models[f"{partition_id}__broad"] = broad_quality_correction
    quality_correction_rows.append(
        {
            "partition_id": partition_id,
            "stage": "broad",
            "parent_broad_type": pd.NA,
            "corrected": broad_quality_correction is not None,
            "quality_column_count": 0 if broad_quality_correction is None else len(broad_quality_correction["quality_columns"]),
            "quality_explained_fraction_before_correction": broad_quality_explained,
        }
    )

    broad_labels, broad_confidence = assign_excluded_regions(
        broad_algorithm, broad_selected, broad_matrix, broad_fit_mask
    )
    training_broad_labels = broad_labels[broad_fit_mask]
    broad_cluster_sizes = {
        int(label): int(np.sum(training_broad_labels == label))
        for label in np.unique(training_broad_labels)
        if label >= 0
    }
    rare_broad_labels = {
        label for label, count in broad_cluster_sizes.items()
        if count < CFG.minimum_broad_regions_per_cluster
    }
    rare_broad_mask = np.array([label in rare_broad_labels for label in broad_labels], dtype=bool)
    regions_df.loc[partition.index[rare_broad_mask], "rare_broad_type"] = True
    broad_names = [
        f"mars_{partition_id}_broad_noise"
        if label < 0
        else f"mars_{partition_id}_broad_{label:02d}" + ("__rare_small" if label in rare_broad_labels else "")
        for label in broad_labels
    ]
    regions_df.loc[partition.index, "broad_local_label"] = broad_labels
    regions_df.loc[partition.index, "broad_cluster_confidence"] = broad_confidence
    regions_df.loc[partition.index, "cluster_confidence"] = broad_confidence
    regions_df.loc[partition.index, "selected_algorithm"] = f"broad={broad_algorithm}"
    regions_df.loc[partition.index, "broad_type"] = broad_names
    regions_df.loc[partition.index, "neutral_type"] = broad_names

    used_groups = set(broad_groups)
    subtype_applied = False
    for broad_name in sorted(set(broad_names)):
        if broad_all_rejected:
            break
        if broad_name.endswith("_noise"):
            continue
        if broad_name.endswith("__rare_small"):
            continue
        subtype_indices = partition.index[np.asarray(broad_names) == broad_name]
        subtype_partition = regions_df.loc[subtype_indices].copy()
        subtype_key = f"{partition_id}__{broad_name.rsplit('_', 1)[-1]}"
        subtype_fit_mask = subtype_partition["model_training_eligible"].to_numpy(dtype=bool)
        if subtype_fit_mask.sum() < CFG.minimum_subtype_regions:
            comparison_frames.append(pd.DataFrame([{
                "partition_id": subtype_key,
                "algorithm": "not_tested",
                "cluster_count": 0,
                "noise_fraction": np.nan,
                "assigned_fraction": np.nan,
                "minimum_cluster_size": 0,
                "silhouette": np.nan,
                "spatial_stability_ari": np.nan,
                "selection_score": -np.inf,
                "criterion": "not tested: too few substantive regions",
                "coverage_balanced_accuracy": np.nan,
                "chance_balanced_accuracy": np.nan,
                "artifact_warning": False,
                "artifact_diagnostic_available": False,
                "quality_gate_pass": False,
                "quality_gate_reason": f"training_regions<{CFG.minimum_subtype_regions}",
                "best_candidate": True,
                "applied_to_output": False,
                "stage": "subtype",
                "parent_broad_type": broad_name,
                "all_candidates_rejected": True,
                "model_training_region_count": int(subtype_fit_mask.sum()),
                "posthoc_assigned_region_count": int((~subtype_fit_mask).sum()),
            }]))
            continue
        try:
            (
                subtype_matrix,
                subtype_columns,
                subtype_column_to_group,
                subtype_groups,
                subtype_quality_correction,
                subtype_quality_explained,
            ) = partition_region_matrix(subtype_partition, manifest, SUBTYPE_GROUPS, subtype_fit_mask)
        except ValueError:
            continue

        subtype_training_partition = subtype_partition.loc[subtype_fit_mask].copy()
        subtype_folds = subtype_training_partition["dominant_spatial_fold"].to_numpy(dtype=np.int16)
        subtype_comparison, subtype_candidates = compare_partition_clusterers(
            subtype_matrix[subtype_fit_mask],
            subtype_folds,
            subtype_key,
            CFG.maximum_subtype_cluster_count,
        )
        subtype_comparison, subtype_algorithm, subtype_selected, subtype_all_rejected = select_candidate_with_artifact_control(
            subtype_comparison,
            subtype_candidates,
            subtype_training_partition,
            subtype_groups,
            stage="subtype",
        )
        subtype_comparison["stage"] = "subtype"
        subtype_comparison["parent_broad_type"] = broad_name
        subtype_comparison["all_candidates_rejected"] = subtype_all_rejected
        subtype_comparison["model_training_region_count"] = int(subtype_fit_mask.sum())
        subtype_comparison["posthoc_assigned_region_count"] = int((~subtype_fit_mask).sum())
        comparison_frames.append(subtype_comparison)
        if subtype_all_rejected:
            continue

        comparison_frames[-1].loc[comparison_frames[-1]["best_candidate"], "applied_to_output"] = True
        subtype_applied = True
        used_groups.update(subtype_groups)
        model_key = f"{subtype_key}__subtype"
        selected_models[model_key] = subtype_selected["model"]
        if subtype_quality_correction is not None:
            quality_correction_models[model_key] = subtype_quality_correction
        quality_correction_rows.append(
            {
                "partition_id": partition_id,
                "stage": "subtype",
                "parent_broad_type": broad_name,
                "corrected": subtype_quality_correction is not None,
                "quality_column_count": 0 if subtype_quality_correction is None else len(subtype_quality_correction["quality_columns"]),
                "quality_explained_fraction_before_correction": subtype_quality_explained,
            }
        )

        subtype_labels, subtype_confidence = assign_excluded_regions(
            subtype_algorithm, subtype_selected, subtype_matrix, subtype_fit_mask
        )
        subtype_names = [
            f"{broad_name}__subtype_noise" if label < 0 else f"{broad_name}__subtype_{label:02d}"
            for label in subtype_labels
        ]
        combined_confidence = np.minimum(
            regions_df.loc[subtype_indices, "broad_cluster_confidence"].to_numpy(dtype=np.float32),
            subtype_confidence,
        )
        regions_df.loc[subtype_indices, "subtype_local_label"] = subtype_labels
        regions_df.loc[subtype_indices, "subtype_cluster_confidence"] = subtype_confidence
        regions_df.loc[subtype_indices, "cluster_confidence"] = combined_confidence
        regions_df.loc[subtype_indices, "selected_algorithm"] = f"broad={broad_algorithm};subtype={subtype_algorithm}"
        regions_df.loc[subtype_indices, "neutral_type"] = subtype_names

    final_names = regions_df.loc[partition.index, "neutral_type"].astype(str)
    final_noise = final_names.str.endswith("_noise")
    final_codes = np.full(len(final_names), -1, dtype=np.int32)
    final_codes[~final_noise.to_numpy()] = pd.factorize(final_names.loc[~final_noise], sort=True)[0].astype(np.int32)
    regions_df.loc[partition.index, "cluster_local_label"] = final_codes
    partition_cluster_context[partition_id] = {
        "matrix": broad_matrix,
        "model_training_mask": broad_fit_mask,
        "region_indices": partition.index.to_numpy(),
        "feature_columns": broad_columns,
        "column_to_group": broad_column_to_group,
        "science_groups": used_groups,
        "candidates": broad_candidates,
        "selected_algorithm": broad_algorithm,
        "broad_all_candidates_rejected": broad_all_rejected,
        "subtype_applied": subtype_applied,
        "quality_explained_fraction_before_correction": broad_quality_explained,
    }

model_comparison_df = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
quality_correction_df = pd.DataFrame(quality_correction_rows)
if regions_df["neutral_type"].isna().any():
    raise ValueError("Some regions did not receive a neutral type")
if len(model_comparison_df):
    if (model_comparison_df["applied_to_output"] & ~model_comparison_df["best_candidate"]).any():
        raise AssertionError("A non-winning clustering candidate was marked as applied")
    unsafe_applied_subtype = (
        (model_comparison_df["stage"] == "subtype")
        & model_comparison_df["applied_to_output"]
        & model_comparison_df["all_candidates_rejected"]
    )
    if unsafe_applied_subtype.any():
        raise AssertionError("An artifact-risk subtype was applied to the output")
if (regions_df.loc[regions_df["cluster_assignment_role"] == "posthoc_tiny", "model_training_eligible"]).any():
    raise AssertionError("A post-hoc tiny fragment was allowed into model fitting")

display(
    model_comparison_df.sort_values(["stage", "partition_id", "selection_score"], ascending=[True, True, False])
    if len(model_comparison_df)
    else pd.DataFrame({"message": ["All partitions were descriptive-only or too small for model comparison"]})
)


## Detect anomalies and test whether clusters are trustworthy

An Isolation Forest supplies an **unusualness score**, not a process label. Separate diagnostics then ask:

1. Can coverage and quality fields predict the neutral type across geographic folds?
2. Does removing one science family radically change a simple K-means baseline?

High artifact predictability is a warning that observation patterns may dominate. Low ablation agreement means a family is influential; this can reflect real geology or an artifact and must be inspected on the maps.

In [ ]:
regions_df["anomaly_score"] = np.nan
regions_df["is_anomaly_candidate"] = False
anomaly_models: dict[str, IsolationForest] = {}

for partition_id, context in partition_cluster_context.items():
    matrix = context["matrix"]
    indices = context["region_indices"]
    fit_mask = np.asarray(context["model_training_mask"], dtype=bool)
    training_matrix = matrix[fit_mask]
    model = IsolationForest(
        n_estimators=300,
        max_samples=min(512, len(training_matrix)),
        contamination="auto",
        n_jobs=-1,
        random_state=CFG.random_seed,
    )
    model.fit(training_matrix)
    scores = -model.decision_function(matrix)
    threshold = float(np.quantile(scores[fit_mask], 1.0 - CFG.anomaly_fraction))
    regions_df.loc[indices, "anomaly_score"] = scores
    regions_df.loc[indices, "is_anomaly_candidate"] = scores >= threshold
    anomaly_models[partition_id] = model
regions_df["is_anomaly_candidate"] |= regions_df["rare_broad_type"]


artifact_rows: list[dict[str, Any]] = []
artifact_importance_rows: list[dict[str, Any]] = []
for partition_id, partition in regions_df.groupby("partition_id", observed=True):
    usable = (partition["cluster_local_label"] >= 0) & partition["model_training_eligible"]
    partition = partition.loc[usable].copy()
    if len(partition) < 20 or partition["cluster_local_label"].nunique() < 2:
        continue
    context = partition_cluster_context.get(str(partition_id))
    if context is None:
        continue
    columns = artifact_feature_columns(partition, set(context["science_groups"]))
    if len(columns) < 1:
        continue
    values = partition[columns].to_numpy(dtype=np.float64)
    labels = partition["cluster_local_label"].to_numpy(dtype=np.int32)
    folds = partition["dominant_spatial_fold"].to_numpy(dtype=np.int16)
    fold_scores: list[float] = []
    importance_values: list[np.ndarray] = []
    for fold in sorted(np.unique(folds)):
        training = folds != fold
        testing = ~training
        if training.sum() < 10 or testing.sum() < 3 or len(np.unique(labels[training])) < 2:
            continue
        imputer = SimpleImputer(strategy="median")
        train_values = imputer.fit_transform(values[training])
        test_values = imputer.transform(values[testing])
        classifier = RandomForestClassifier(
            n_estimators=250,
            class_weight="balanced_subsample",
            min_samples_leaf=max(2, int(training.sum() // 500)),
            n_jobs=-1,
            random_state=CFG.random_seed + int(fold),
        )
        classifier.fit(train_values, labels[training])
        predictions = classifier.predict(test_values)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=UserWarning)
            fold_scores.append(float(balanced_accuracy_score(labels[testing], predictions)))
        importance_values.append(classifier.feature_importances_)

    if not fold_scores:
        continue
    score = float(np.mean(fold_scores))
    class_count = int(partition["cluster_local_label"].nunique())
    chance_balanced_accuracy = 1.0 / class_count
    warning = score >= max(CFG.artifact_warning_balanced_accuracy, chance_balanced_accuracy + 0.25)
    artifact_rows.append(
        {
            "partition_id": partition_id,
            "region_count": len(partition),
            "cluster_count": class_count,
            "coverage_balanced_accuracy": score,
            "chance_balanced_accuracy": chance_balanced_accuracy,
            "artifact_warning": warning,
        }
    )
    mean_importance = np.mean(importance_values, axis=0)
    for column, importance in zip(columns, mean_importance):
        artifact_importance_rows.append(
            {"partition_id": partition_id, "feature": column, "importance": float(importance)}
        )

artifact_diagnostics_df = pd.DataFrame(
    artifact_rows,
    columns=[
        "partition_id", "region_count", "cluster_count",
        "coverage_balanced_accuracy", "chance_balanced_accuracy", "artifact_warning",
    ],
)
artifact_importance_df = pd.DataFrame(
    artifact_importance_rows, columns=["partition_id", "feature", "importance"]
)
artifact_warning_lookup = (
    artifact_diagnostics_df.set_index("partition_id")["artifact_warning"].to_dict()
    if len(artifact_diagnostics_df)
    else {}
)
artifact_available_partitions = set(artifact_diagnostics_df["partition_id"]) if len(artifact_diagnostics_df) else set()
regions_df["artifact_diagnostic_available"] = regions_df["partition_id"].isin(artifact_available_partitions)
regions_df["artifact_warning"] = regions_df["partition_id"].map(artifact_warning_lookup).eq(True)
minimum_cells_by_scale = {
    "medium_1deg": CFG.minimum_medium_region_cells,
    "coarse_10deg": CFG.minimum_coarse_region_cells,
}
regions_df["is_reviewable_anomaly"] = (
    regions_df["is_anomaly_candidate"]
    & ~regions_df["limited_feature_evidence"]
    & ~regions_df["coverage_fragmented"]
    & regions_df["artifact_diagnostic_available"]
    & ~regions_df["artifact_warning"]
    & regions_df["cell_count"].ge(regions_df["scale"].map(minimum_cells_by_scale))
)

ablation_rows: list[dict[str, Any]] = []
for partition_id, context in partition_cluster_context.items():
    fit_mask = np.asarray(context["model_training_mask"], dtype=bool)
    matrix = context["matrix"][fit_mask]
    columns = context["feature_columns"]
    column_to_group = context["column_to_group"]
    kmeans_candidate = context["candidates"]["kmeans"]
    baseline_labels = np.asarray(kmeans_candidate["labels"])
    cluster_count = int(kmeans_candidate["cluster_count"])
    groups = sorted({column_to_group[column] for column in columns})
    for removed_group in groups:
        keep = np.array([column_to_group[column] != removed_group for column in columns])
        if keep.sum() < 2:
            continue
        ablated_model = KMeans(n_clusters=cluster_count, n_init=20, random_state=CFG.random_seed)
        ablated_labels = ablated_model.fit_predict(matrix[:, keep])
        ablation_rows.append(
            {
                "partition_id": partition_id,
                "removed_group": removed_group,
                "remaining_feature_count": int(keep.sum()),
                "agreement_ari": float(adjusted_rand_score(baseline_labels, ablated_labels)),
                "influence": float(1.0 - adjusted_rand_score(baseline_labels, ablated_labels)),
            }
        )

ablation_df = pd.DataFrame(ablation_rows)
display(artifact_diagnostics_df if len(artifact_diagnostics_df) else pd.DataFrame({"message": ["No partition supported an artifact classifier"]}))
display(ablation_df.sort_values("influence", ascending=False).head(15) if len(ablation_df) else pd.DataFrame({"message": ["No ablation tests available"]}))

## Describe neutral types as distributions before interpreting them

A mean or percentile is useful for a quick human-readable profile, but it is not a process signature. This section therefore also preserves weighted quantiles, spread, missingness, feature-to-feature dependence, and spatial structure.

The resulting tables describe **neutral types**, not volcanic, aqueous, aeolian, or ice-related processes. After independent process validation, the same table format can be rebuilt from validated examples and compared with harmonized Earth distributions.


In [ ]:
PROFILE_FEATURES = [
    "elevation_mean_m",
    "relief_m",
    "slope_mean_deg",
    "thermal_inertia_mean",
    "albedo_mean",
    "quality_masked__mineral_surface_dust",
    "detection_supported__mineral_plagioclase",
    "detection_supported__mineral_low_ca_pyroxene",
    "detection_supported__mineral_high_ca_pyroxene",
    "detection_supported__mineral_olivine",
    "detection_supported__mineral_hematite",
    "detection_supported__mineral_sulfates",
    "detection_supported__mineral_carbonates",
    "detection_supported__mineral_sheet_silicates_high_si_glass",
    "emissivity_spectral_contrast",
    "neutron_epithermal_counts_s",
    "neutron_thermal_counts_s",
    "neutron_fast_counts_s",
]

SIGNATURE_QUANTILES = (0.10, 0.25, 0.50, 0.75, 0.90)


def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    valid = values.notna() & weights.notna() & (weights > 0)
    if not valid.any():
        return float("nan")
    return float(np.average(values[valid], weights=weights[valid]))


def weighted_quantiles(values: np.ndarray, weights: np.ndarray, quantiles: tuple[float, ...]) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.full(len(quantiles), np.nan, dtype=np.float64)
    values = values[valid]
    weights = weights[valid]
    order = np.argsort(values, kind="mergesort")
    values = values[order]
    weights = weights[order]
    positions = (np.cumsum(weights) - 0.5 * weights) / weights.sum()
    return np.interp(np.asarray(quantiles), positions, values, left=values[0], right=values[-1])


def weighted_location_scale(values: np.ndarray, weights: np.ndarray) -> tuple[float, float]:
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return float("nan"), float("nan")
    values = values[valid]
    weights = weights[valid]
    mean = float(np.average(values, weights=weights))
    variance = float(np.average(np.square(values - mean), weights=weights))
    return mean, math.sqrt(max(variance, 0.0))


profile_rows: list[dict[str, Any]] = []
cluster_summary_rows: list[dict[str, Any]] = []
for (partition_id, neutral_type), cluster_regions in regions_df.groupby(["partition_id", "neutral_type"], observed=True):
    cluster_summary_rows.append(
        {
            "partition_id": partition_id,
            "neutral_type": neutral_type,
            "scale": cluster_regions["scale"].iloc[0],
            "analysis_cohort": cluster_regions["analysis_cohort"].iloc[0],
            "limited_feature_evidence": bool(cluster_regions["limited_feature_evidence"].iloc[0]),
            "coverage_fragmented": bool(cluster_regions["coverage_fragmented"].iloc[0]),
            "artifact_warning": bool(cluster_regions["artifact_warning"].iloc[0]),
            "region_count": int(len(cluster_regions)),
            "area_km2": float(cluster_regions["area_km2"].sum()),
            "anomaly_region_fraction": float(cluster_regions["is_anomaly_candidate"].mean()),
            "reviewable_anomaly_fraction": float(cluster_regions["is_reviewable_anomaly"].mean()),
            "median_cluster_confidence": float(cluster_regions["cluster_confidence"].median()),
        }
    )
    partition_regions = regions_df.loc[regions_df["partition_id"] == partition_id]
    for feature in PROFILE_FEATURES:
        column = f"raw_mean__{feature}"
        if column not in regions_df.columns or cluster_regions[column].notna().sum() == 0:
            continue
        value = weighted_mean(cluster_regions[column], cluster_regions["area_km2"])
        reference = partition_regions[column].dropna()
        percentile = float((reference <= value).mean()) if np.isfinite(value) and len(reference) else float("nan")
        profile_rows.append(
            {
                "partition_id": partition_id,
                "neutral_type": neutral_type,
                "feature": feature,
                "value": value,
                "partition_percentile": percentile,
            }
        )
    for feature in [name for name in PROFILE_FEATURES if name.startswith("detection_supported__")]:
        column = f"raw_positive_fraction__{feature}"
        if column not in regions_df.columns or cluster_regions[column].notna().sum() == 0:
            continue
        value = weighted_mean(cluster_regions[column], cluster_regions["area_km2"])
        reference = partition_regions[column].dropna()
        percentile = float((reference <= value).mean()) if np.isfinite(value) and len(reference) else float("nan")
        profile_rows.append(
            {
                "partition_id": partition_id,
                "neutral_type": neutral_type,
                "feature": f"positive_fraction__{feature}",
                "value": value,
                "partition_percentile": percentile,
            }
        )

cluster_summary_df = pd.DataFrame(cluster_summary_rows)
cluster_profiles_df = pd.DataFrame(profile_rows)

# Join neutral labels back to original cells. Physical measurements are used for
# marginal distributions; model-space values are used only for dependence estimates.
signature_cells_df = region_cells_df.merge(
    regions_df[["region_id", "partition_id", "neutral_type"]],
    on="region_id",
    how="left",
    validate="many_to_one",
)

distribution_rows: list[dict[str, Any]] = []
region_distribution_rows: list[dict[str, Any]] = []
dependency_rows: list[dict[str, Any]] = []
spatial_rows: list[dict[str, Any]] = []

for (partition_id, neutral_type), type_cells in signature_cells_df.groupby(
    ["partition_id", "neutral_type"], observed=True
):
    scale_name = str(type_cells["scale"].iloc[0])
    science_features = active_schema_df.loc[active_schema_df["scale"] == scale_name, "feature"].tolist()
    total_area = float(type_cells["cell_area_km2"].sum())
    cell_weights = type_cells["cell_area_km2"].to_numpy(dtype=np.float64)

    for feature in science_features:
        if feature not in type_cells.columns:
            continue
        values = pd.to_numeric(type_cells[feature], errors="coerce").to_numpy(dtype=np.float64)
        valid = np.isfinite(values) & np.isfinite(cell_weights) & (cell_weights > 0)
        if not valid.any():
            continue
        quantile_values = weighted_quantiles(values, cell_weights, SIGNATURE_QUANTILES)
        mean, standard_deviation = weighted_location_scale(values, cell_weights)
        valid_area = float(cell_weights[valid].sum())
        distribution_rows.append(
            {
                "signature_scope": "neutral_type_not_process",
                "partition_id": partition_id,
                "neutral_type": neutral_type,
                "scale": scale_name,
                "feature": feature,
                "finite_cell_count": int(valid.sum()),
                "finite_area_fraction": valid_area / total_area if total_area > 0 else float("nan"),
                "weighted_mean": mean,
                "weighted_std": standard_deviation,
                "q10": float(quantile_values[0]),
                "q25": float(quantile_values[1]),
                "q50": float(quantile_values[2]),
                "q75": float(quantile_values[3]),
                "q90": float(quantile_values[4]),
            }
        )

    manifest_row = partition_manifest_df.loc[partition_manifest_df["partition_id"] == partition_id].iloc[0]
    selected_features = [
        feature for feature in manifest_row["selected_features"] if f"model__{feature}" in type_cells.columns
    ]
    for left_index, left_feature in enumerate(selected_features):
        left_values = pd.to_numeric(type_cells[f"model__{left_feature}"], errors="coerce").to_numpy(dtype=np.float64)
        for right_feature in selected_features[left_index + 1 :]:
            right_values = pd.to_numeric(type_cells[f"model__{right_feature}"], errors="coerce").to_numpy(dtype=np.float64)
            valid = (
                np.isfinite(left_values)
                & np.isfinite(right_values)
                & np.isfinite(cell_weights)
                & (cell_weights > 0)
            )
            if valid.sum() < 3:
                continue
            weights = cell_weights[valid]
            left = left_values[valid]
            right = right_values[valid]
            left_mean = np.average(left, weights=weights)
            right_mean = np.average(right, weights=weights)
            covariance = float(np.average((left - left_mean) * (right - right_mean), weights=weights))
            left_variance = float(np.average(np.square(left - left_mean), weights=weights))
            right_variance = float(np.average(np.square(right - right_mean), weights=weights))
            denominator = math.sqrt(max(left_variance * right_variance, 0.0))
            dependency_rows.append(
                {
                    "signature_scope": "neutral_type_not_process",
                    "partition_id": partition_id,
                    "neutral_type": neutral_type,
                    "scale": scale_name,
                    "feature_left": left_feature,
                    "feature_right": right_feature,
                    "weighted_covariance_model_space": covariance,
                    "weighted_correlation_model_space": covariance / denominator if denominator > 1e-12 else float("nan"),
                    "common_cell_count": int(valid.sum()),
                    "common_area_fraction": float(weights.sum() / total_area) if total_area > 0 else float("nan"),
                }
            )

    type_regions = regions_df.loc[regions_df["neutral_type"] == neutral_type]
    for feature in ["area_km2", "cell_count", "perimeter_km", "compactness", "cluster_confidence", "anomaly_score"]:
        values = pd.to_numeric(type_regions[feature], errors="coerce").to_numpy(dtype=np.float64)
        weights = type_regions["area_km2"].to_numpy(dtype=np.float64)
        valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
        if not valid.any():
            continue
        quantile_values = weighted_quantiles(values, weights, SIGNATURE_QUANTILES)
        spatial_rows.append(
            {
                "signature_scope": "neutral_type_not_process",
                "partition_id": partition_id,
                "neutral_type": neutral_type,
                "feature": feature,
                "region_count": int(valid.sum()),
                "q10": float(quantile_values[0]),
                "q25": float(quantile_values[1]),
                "q50": float(quantile_values[2]),
                "q75": float(quantile_values[3]),
                "q90": float(quantile_values[4]),
            }
        )

# Preserve region-level distributions too. The later validation stage can select
# independently confirmed regions without reducing each one to a single mean.
for region_id, region_cells in signature_cells_df.groupby("region_id", observed=True):
    scale_name = str(region_cells["scale"].iloc[0])
    science_features = active_schema_df.loc[active_schema_df["scale"] == scale_name, "feature"].tolist()
    total_area = float(region_cells["cell_area_km2"].sum())
    cell_weights = region_cells["cell_area_km2"].to_numpy(dtype=np.float64)
    for feature in science_features:
        if feature not in region_cells.columns:
            continue
        values = pd.to_numeric(region_cells[feature], errors="coerce").to_numpy(dtype=np.float64)
        valid = np.isfinite(values) & np.isfinite(cell_weights) & (cell_weights > 0)
        if not valid.any():
            continue
        quantile_values = weighted_quantiles(values, cell_weights, SIGNATURE_QUANTILES)
        mean, standard_deviation = weighted_location_scale(values, cell_weights)
        region_distribution_rows.append(
            {
                "signature_scope": "region_not_process",
                "region_id": region_id,
                "partition_id": region_cells["partition_id"].iloc[0],
                "neutral_type": region_cells["neutral_type"].iloc[0],
                "scale": scale_name,
                "feature": feature,
                "finite_cell_count": int(valid.sum()),
                "finite_area_fraction": float(cell_weights[valid].sum() / total_area) if total_area > 0 else float("nan"),
                "weighted_mean": mean,
                "weighted_std": standard_deviation,
                "q10": float(quantile_values[0]),
                "q25": float(quantile_values[1]),
                "q50": float(quantile_values[2]),
                "q75": float(quantile_values[3]),
                "q90": float(quantile_values[4]),
            }
        )

distribution_columns = [
    "signature_scope", "partition_id", "neutral_type", "scale", "feature",
    "finite_cell_count", "finite_area_fraction", "weighted_mean", "weighted_std",
    "q10", "q25", "q50", "q75", "q90",
]
region_feature_distributions_df = pd.DataFrame(
    region_distribution_rows, columns=["region_id", *distribution_columns]
)
neutral_type_distributions_df = pd.DataFrame(distribution_rows, columns=distribution_columns)
neutral_type_dependencies_df = pd.DataFrame(
    dependency_rows,
    columns=[
        "signature_scope", "partition_id", "neutral_type", "scale", "feature_left",
        "feature_right", "weighted_covariance_model_space",
        "weighted_correlation_model_space", "common_cell_count", "common_area_fraction",
    ],
)
neutral_type_spatial_signatures_df = pd.DataFrame(
    spatial_rows,
    columns=[
        "signature_scope", "partition_id", "neutral_type", "feature", "region_count",
        "q10", "q25", "q50", "q75", "q90",
    ],
)

display(cluster_summary_df.sort_values("area_km2", ascending=False).head(20))
display(neutral_type_distributions_df.head(20))


## Audit process hypotheses against paper-defined evidence

This is an **evidence audit**, not a classifier. No percentile threshold below is invented and no region receives a process name in this notebook.

What the cited work supports:

- Statistical mineral assemblages identify compositional provinces, not their cause by themselves: [Rogers & Hamilton (2015)](https://doi.org/10.1002/2014JE004690).
- A volcanic interpretation uses geology, stratigraphy, morphology, and topography together; mafic composition alone is insufficient: [Hiesinger & Head (2004)](https://doi.org/10.1029/2003JE002143).
- Aqueous-mineral interpretations depend on mineral assemblage, morphology, and geological setting: [Murchie et al. (2009)](https://doi.org/10.1029/2009JE003342), [Weitz et al. (2008)](https://doi.org/10.1029/2007JE003070).
- Active aeolian transport requires measured bedform change from repeat observations: [Bridges et al. (2012)](https://doi.org/10.1038/nature11022).
- Low epithermal-neutron flux supports hydrogen enrichment, not a unique material or process: [Feldman et al. (2002)](https://doi.org/10.1126/science.1073541).

The registry explicitly records what the current global products can contribute and what must come from targeted imagery, spectroscopy, radar, seasonal measurements, or repeat observations.


In [ ]:
PROCESS_HYPOTHESES: list[dict[str, Any]] = [
    {
        "hypothesis_id": "past_volcanic_emplacement",
        "target_kind": "past_process",
        "claim": "Material was emplaced by volcanic activity.",
        "required_evidence": [
            "volcanic_landform_or_flow_morphology",
            "geological_or_stratigraphic_association",
        ],
        "supporting_evidence": [
            "mafic_or_igneous_compositional_assemblage",
            "topographic_continuity",
            "rock_or_indurated_surface_context",
        ],
        "available_in_current_global_products": [
            "broad_compositional_assemblage",
            "coarse_topography",
            "thermal_inertia_context",
        ],
        "not_available_in_current_global_products": [
            "diagnostic_flow_or_edifice_morphology",
            "mapped_unit_stratigraphy",
        ],
        "alternatives_to_test": [
            "impact-excavated igneous crust",
            "transported mafic sediment",
            "non-volcanic igneous exposure",
        ],
        "sources": [
            "https://doi.org/10.1002/2014JE004690",
            "https://doi.org/10.1029/2003JE002143",
        ],
    },
    {
        "hypothesis_id": "water_driven_mineral_alteration",
        "target_kind": "past_process",
        "claim": "Minerals record alteration involving liquid water or hydrothermal fluids.",
        "required_evidence": [
            "validated_secondary_mineral_identification",
            "geological_association_with_the_mapped_unit",
        ],
        "supporting_evidence": [
            "compatible_mineral_assemblage",
            "fluvial_lacustrine_hydrothermal_or_evaporitic_context",
            "stratigraphic_relationships",
        ],
        "available_in_current_global_products": [
            "broad_secondary_mineral_screening",
            "coarse_topography",
            "thermal_inertia_context",
        ],
        "not_available_in_current_global_products": [
            "high_resolution_mineral_validation",
            "diagnostic_morphology_and_stratigraphy",
        ],
        "alternatives_to_test": [
            "thermal oxidation",
            "volcanic glass or ambiguous high-silica phase",
            "transported mineral-bearing sediment",
        ],
        "sources": [
            "https://doi.org/10.1029/2009JE003342",
            "https://doi.org/10.1029/2007JE003070",
        ],
    },
    {
        "hypothesis_id": "active_aeolian_transport",
        "target_kind": "current_process",
        "claim": "Wind is currently moving surface sediment.",
        "required_evidence": [
            "aeolian_bedform_or_wind_streak_morphology",
            "repeat_observation_change_beyond_registration_error",
        ],
        "supporting_evidence": [
            "fine_or_unconsolidated_surface_context",
            "wind_or_atmospheric_dust_context",
            "consistent_transport_orientation",
        ],
        "available_in_current_global_products": [
            "dust_component",
            "thermal_inertia_context",
            "albedo_context",
        ],
        "not_available_in_current_global_products": [
            "resolved_bedform_morphology",
            "repeat_change_measurement",
            "wind_transport_orientation",
        ],
        "alternatives_to_test": [
            "static dust mantle",
            "fine sediment unrelated to current transport",
            "porous or low-density material",
        ],
        "sources": ["https://doi.org/10.1038/nature11022"],
    },
    {
        "hypothesis_id": "ice_related_modification_or_storage",
        "target_kind": "process_or_environment",
        "claim": "Ice is stored in or has modified the shallow subsurface or surface.",
        "required_evidence": [
            "hydrogen_or_ice_sensitive_measurement",
            "independent_landform_radar_thermal_or_spectral_context",
        ],
        "supporting_evidence": [
            "latitude_elevation_and_seasonal_context",
            "compatible_landform_assemblage",
        ],
        "available_in_current_global_products": [
            "coarse_hydrogen_sensitive_neutron_measurement",
            "coarse_topography",
        ],
        "not_available_in_current_global_products": [
            "material_identification_distinguishing_ice_from_bound_hydrogen",
            "radar_or_diagnostic_landform_confirmation",
            "seasonal_or_temporal_behavior",
        ],
        "alternatives_to_test": [
            "hydrated minerals",
            "chemically bound hydrogen",
            "composition-related neutron response",
        ],
        "sources": ["https://doi.org/10.1126/science.1073541"],
    },
]


def process_audit_row(hypothesis: dict[str, Any]) -> dict[str, Any]:
    missing = list(hypothesis["not_available_in_current_global_products"])
    return {
        "hypothesis_id": hypothesis["hypothesis_id"],
        "target_kind": hypothesis["target_kind"],
        "claim": hypothesis["claim"],
        "current_global_contribution": "; ".join(hypothesis["available_in_current_global_products"]),
        "missing_required_or_disambiguating_evidence": "; ".join(missing),
        "global_stage_status": "candidate_selection_only_not_process_validated",
        "may_build_earth_process_signature_now": False,
        "sources": hypothesis["sources"],
    }


process_hypothesis_audit_df = pd.DataFrame([process_audit_row(item) for item in PROCESS_HYPOTHESES])
display(
    process_hypothesis_audit_df[
        [
            "hypothesis_id",
            "target_kind",
            "global_stage_status",
            "missing_required_or_disambiguating_evidence",
        ]
    ]
)


## Reference locations: sanity checks only

Known locations help us inspect whether neutral types behave sensibly. They are not training labels, they do not label every member of a cluster, and they do not validate a process distribution. A future validation stage needs multiple positive examples, negative controls, and the detailed observations required by the relevant papers.


In [ ]:
CONTROL_SITES = [
    {"name": "Syrtis Major", "latitude": 9.5, "longitude": 69.5, "preferred_scale": "medium_1deg", "known_context": "Volcanic province used only as a map-orientation check", "source": "https://doi.org/10.1029/2003JE002143"},
    {"name": "Nili Fossae", "latitude": 22.0, "longitude": 76.0, "preferred_scale": "medium_1deg", "known_context": "Compositionally complex reference area; not a single-process label", "source": "https://doi.org/10.1002/2014JE004690"},
    {"name": "Meridiani Planum", "latitude": -2.0, "longitude": -5.5, "preferred_scale": "medium_1deg", "known_context": "Hematite/sulfate-bearing sedimentary and alteration context", "source": "https://doi.org/10.1029/2009JE003342"},
    {"name": "Olympus Mons", "latitude": 18.65, "longitude": -133.8, "preferred_scale": "medium_1deg", "known_context": "Large volcanic construct used only as a map-orientation check", "source": "https://doi.org/10.1029/2002JE002031"},
    {"name": "Medusae Fossae", "latitude": 0.0, "longitude": -155.0, "preferred_scale": "medium_1deg", "known_context": "Fine-grained, morphologically complex reference area with debated origin", "source": "https://doi.org/10.1029/2008JE003076"},
    {"name": "North polar terrain", "latitude": 80.0, "longitude": 0.0, "preferred_scale": "coarse_10deg", "known_context": "High-latitude hydrogen-sensitive neutron reference", "source": "https://doi.org/10.1126/science.1073541"},
    {"name": "South polar terrain", "latitude": -80.0, "longitude": 0.0, "preferred_scale": "coarse_10deg", "known_context": "High-latitude hydrogen-sensitive neutron reference", "source": "https://doi.org/10.1126/science.1073541"},
]


def angular_distance_degrees(latitudes: np.ndarray, longitudes: np.ndarray, latitude: float, longitude: float) -> np.ndarray:
    lat1 = np.deg2rad(latitudes)
    lon1 = np.deg2rad(longitudes)
    lat2 = math.radians(latitude)
    lon2 = math.radians(longitude)
    cosine = np.sin(lat1) * math.sin(lat2) + np.cos(lat1) * math.cos(lat2) * np.cos(lon1 - lon2)
    return np.rad2deg(np.arccos(np.clip(cosine, -1.0, 1.0)))


control_rows: list[dict[str, Any]] = []
for site in CONTROL_SITES:
    candidates = region_cells_df.loc[region_cells_df["scale"] == site["preferred_scale"]]
    distances = angular_distance_degrees(
        candidates["latitude"].to_numpy(dtype=float),
        candidates["longitude"].to_numpy(dtype=float),
        site["latitude"],
        site["longitude"],
    )
    nearest_index = candidates.index[int(np.argmin(distances))]
    nearest_cell = candidates.loc[nearest_index]
    region = regions_df.loc[regions_df["region_id"] == nearest_cell["region_id"]].iloc[0]
    control_rows.append(
        {
            **site,
            "nearest_cell_id": nearest_cell["cell_id"],
            "angular_distance_degrees": float(np.min(distances)),
            "region_id": region["region_id"],
            "neutral_type": region["neutral_type"],
            "selected_algorithm": region["selected_algorithm"],
            "cluster_confidence": region["cluster_confidence"],
            "is_anomaly_candidate": bool(region["is_anomaly_candidate"]),
            "is_reviewable_anomaly": bool(region["is_reviewable_anomaly"]),
            "artifact_warning": bool(region["artifact_warning"]),
        }
    )

control_sites_df = pd.DataFrame(control_rows)
display(control_sites_df)


## Visual analysis dashboard

Each figure answers one decision:

1. Are region boundaries spatially coherent?
2. Which clustering behavior is stable?
3. Where do the neutral types occur and what defines them?
4. Are types predictable from coverage, or controlled by one feature family?
5. Where are unusual regions, and where are neutral signatures sufficiently rich for later validation?

No figure in this notebook is a process map.


In [ ]:
region_result_columns = [
    "region_id",
    "broad_type",
    "neutral_type",
    "selected_algorithm",
    "cluster_confidence",
    "broad_cluster_confidence",
    "subtype_cluster_confidence",
    "limited_feature_evidence",
    "coverage_fragmented",
    "model_training_eligible",
    "cluster_assignment_role",
    "rare_broad_type",
    "artifact_diagnostic_available",
    "artifact_warning",
    "anomaly_score",
    "is_anomaly_candidate",
    "is_reviewable_anomaly",
]
region_cell_results_df = region_cells_df.merge(
    regions_df[region_result_columns],
    on="region_id",
    how="left",
    validate="many_to_one",
)


def raster_from_cells(frame: pd.DataFrame, value_column: str, shape: tuple[int, int], fill_value: float = np.nan) -> np.ndarray:
    raster = np.full(shape, fill_value, dtype=np.float64)
    raster[frame["row"].to_numpy(dtype=np.int32), frame["column"].to_numpy(dtype=np.int32)] = frame[value_column].to_numpy()
    return raster


def boundary_mask(labels: np.ndarray) -> np.ndarray:
    boundary = labels != np.roll(labels, -1, axis=1)
    boundary[:-1, :] |= labels[:-1, :] != labels[1:, :]
    return boundary


medium_cells = region_cell_results_df.loc[region_cell_results_df["scale"] == "medium_1deg"].copy()
medium_shape = grid_shape(medium_cells)
medium_labels = scale_label_rasters["medium_1deg"]
medium_elevation = raster_from_cells(medium_cells, "elevation_mean_m", medium_shape)
medium_boundaries = boundary_mask(medium_labels)

fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)
elevation_limits = np.nanpercentile(medium_elevation, [2, 98])
image = axes[0].imshow(
    medium_elevation,
    extent=[-180, 180, -90, 90],
    origin="upper",
    aspect="auto",
    cmap="terrain",
    vmin=elevation_limits[0],
    vmax=elevation_limits[1],
)
axes[0].imshow(
    np.where(medium_boundaries, 1.0, np.nan),
    extent=[-180, 180, -90, 90],
    origin="upper",
    aspect="auto",
    cmap=mcolors.ListedColormap(["#111827"]),
    alpha=0.35,
    interpolation="nearest",
)
axes[0].set_title("Connected 1° region boundaries over elevation", loc="left", fontweight="bold")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
plt.colorbar(image, ax=axes[0], fraction=0.025, pad=0.02, label="elevation (m)")

medium_region_sizes = regions_df.loc[regions_df["scale"] == "medium_1deg", "cell_count"].to_numpy(dtype=float)
size_bins = np.geomspace(1.0, max(2.0, float(medium_region_sizes.max()) + 1.0), 36)
axes[1].hist(medium_region_sizes, bins=size_bins, color="#2563EB", edgecolor="white", linewidth=0.4)
axes[1].set_xscale("log")
if np.histogram(medium_region_sizes, bins=size_bins)[0].max() > 1:
    axes[1].set_yscale("log")
axes[1].axvline(
    CFG.minimum_medium_region_cells,
    color="#DC2626",
    linestyle="--",
    label="excluded from fitting; assigned afterward",
)
axes[1].set_title("Region-size distribution", loc="left", fontweight="bold")
axes[1].set_xlabel("1° cells per connected region (log scale)")
axes[1].set_ylabel("Region count (log scale)")
axes[1].legend()
plt.show()


In [ ]:
if len(model_comparison_df):
    broad_plot = model_comparison_df.loc[model_comparison_df["stage"] == "broad"].copy()
    broad_plot["partition_short"] = broad_plot["partition_id"].str.replace("medium_1deg__", "M: ", regex=False).str.replace("coarse_10deg__", "C: ", regex=False)
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(18, max(5, 0.55 * broad_plot["partition_short"].nunique())),
        constrained_layout=True,
    )
    sns.barplot(
        data=broad_plot,
        x="selection_score",
        y="partition_short",
        hue="algorithm",
        orient="h",
        ax=axes[0],
    )
    axes[0].set_xlim(0, 1)
    axes[0].set_xlabel("Geographic stability, separation, and assigned coverage")
    axes[0].set_ylabel("")
    axes[0].set_title("Broad-type clusterer comparison", loc="left", fontweight="bold")

    subtype_decisions = model_comparison_df.loc[
        (model_comparison_df["stage"] == "subtype") & model_comparison_df["best_candidate"]
    ].copy()
    if len(subtype_decisions):
        subtype_decisions["outcome"] = np.where(
            subtype_decisions["applied_to_output"],
            "accepted and applied subtype",
            "rejected: not applied",
        )
        outcome_counts = (
            subtype_decisions["outcome"].value_counts().rename_axis("outcome").reset_index(name="broad_types")
        )
        sns.barplot(
            data=outcome_counts,
            x="broad_types",
            y="outcome",
            hue="outcome",
            legend=False,
            ax=axes[1],
        )
        axes[1].set_xlabel("Number of broad types tested")
        axes[1].set_ylabel("")
    else:
        axes[1].text(
            0.5,
            0.5,
            "No broad type had enough rich measurements for subtyping",
            ha="center",
            va="center",
            wrap=True,
        )
        axes[1].set_axis_off()
    axes[1].set_title(
        "Did richer measurements produce trustworthy subtypes?",
        loc="left",
        fontweight="bold",
    )
    plt.show()


In [ ]:
neutral_type_codes, neutral_type_names = pd.factorize(medium_cells["neutral_type"], sort=True)
medium_cells["neutral_type_code"] = neutral_type_codes
cluster_raster = raster_from_cells(medium_cells, "neutral_type_code", medium_shape, fill_value=-1)

fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)
axes[0].imshow(
    cluster_raster,
    extent=[-180, 180, -90, 90],
    origin="upper",
    aspect="auto",
    cmap="turbo",
    interpolation="nearest",
)
axes[0].set_title("Neutral 1° region types", loc="left", fontweight="bold")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

largest_types = (
    cluster_summary_df.loc[cluster_summary_df["scale"] == "medium_1deg"]
    .nlargest(20, "area_km2")["neutral_type"]
    .tolist()
)
key_profile_features = [
    "elevation_mean_m",
    "relief_m",
    "slope_mean_deg",
    "thermal_inertia_mean",
    "albedo_mean",
    "quality_masked__mineral_surface_dust",
    "detection_supported__mineral_high_ca_pyroxene",
    "detection_supported__mineral_hematite",
    "emissivity_spectral_contrast",
]
profile_heatmap = (
    cluster_profiles_df.loc[
        cluster_profiles_df["neutral_type"].isin(largest_types)
        & cluster_profiles_df["feature"].isin(key_profile_features)
    ]
    .pivot(index="neutral_type", columns="feature", values="partition_percentile")
    .reindex(index=largest_types)
)
if profile_heatmap.notna().any().any():
    sns.heatmap(
        profile_heatmap,
        cmap="vlag",
        center=0.5,
        vmin=0,
        vmax=1,
        ax=axes[1],
        cbar_kws={"label": "percentile within comparable cohort"},
    )
    axes[1].set_title("Largest neutral types — defining properties", loc="left", fontweight="bold")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("")
else:
    axes[1].text(0.5, 0.5, "No common profile features", ha="center", va="center")
    axes[1].set_axis_off()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)
if len(artifact_diagnostics_df):
    artifact_plot = artifact_diagnostics_df.sort_values("coverage_balanced_accuracy", ascending=False)
    sns.barplot(data=artifact_plot, x="coverage_balanced_accuracy", y="partition_id", color="#F59E0B", ax=axes[0])
    axes[0].axvline(CFG.artifact_warning_balanced_accuracy, color="#DC2626", linestyle="--", label="warning threshold")
    axes[0].set_xlim(0, 1)
    axes[0].set_title("Can coverage/quality predict the neutral type?", loc="left", fontweight="bold")
    axes[0].set_xlabel("Spatial-fold balanced accuracy")
    axes[0].set_ylabel("")
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, "Artifact diagnostic unavailable", ha="center", va="center")
    axes[0].set_axis_off()

if len(ablation_df):
    ablation_plot = ablation_df.nlargest(20, "influence")
    sns.barplot(data=ablation_plot, x="influence", y="removed_group", hue="partition_id", dodge=False, ax=axes[1])
    axes[1].set_xlim(0, 1)
    axes[1].set_title("Which broad feature-family removals change K-means most?", loc="left", fontweight="bold")
    axes[1].set_xlabel("Influence = 1 − adjusted Rand agreement")
    axes[1].set_ylabel("")
    axes[1].legend(fontsize=8, title="partition", bbox_to_anchor=(1.02, 1), loc="upper left")
else:
    axes[1].text(0.5, 0.5, "Ablation diagnostic unavailable", ha="center", va="center")
    axes[1].set_axis_off()
plt.show()

In [ ]:
anomaly_raster = raster_from_cells(medium_cells, "anomaly_score", medium_shape)

signature_richness = (
    neutral_type_distributions_df.loc[neutral_type_distributions_df["finite_area_fraction"] >= 0.80]
    .groupby("neutral_type", observed=True)["feature"]
    .nunique()
    .rename("signature_feature_count")
)
medium_cells["signature_feature_count"] = medium_cells["neutral_type"].map(signature_richness).fillna(0).astype(int)
signature_richness_raster = raster_from_cells(medium_cells, "signature_feature_count", medium_shape, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)
anomaly_limits = np.nanpercentile(anomaly_raster, [2, 98])
anomaly_image = axes[0].imshow(
    anomaly_raster,
    extent=[-180, 180, -90, 90],
    origin="upper",
    aspect="auto",
    cmap="magma",
    vmin=anomaly_limits[0],
    vmax=anomaly_limits[1],
)
anomaly_regions = regions_df.loc[
    (regions_df["scale"] == "medium_1deg") & regions_df["is_reviewable_anomaly"]
]
axes[0].scatter(
    anomaly_regions["centroid_longitude"],
    anomaly_regions["centroid_latitude"],
    s=16,
    facecolors="none",
    edgecolors="#22D3EE",
    linewidths=0.8,
    label="reviewable anomaly candidates",
)
axes[0].set_title("Unusual-region score", loc="left", fontweight="bold")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].legend(loc="lower left")
plt.colorbar(anomaly_image, ax=axes[0], fraction=0.025, pad=0.02, label="unusualness")

richness_image = axes[1].imshow(
    signature_richness_raster,
    extent=[-180, 180, -90, 90],
    origin="upper",
    aspect="auto",
    cmap="viridis",
    interpolation="nearest",
)
axes[1].set_title("Neutral-signature richness (not process confidence)", loc="left", fontweight="bold")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
plt.colorbar(richness_image, ax=axes[1], fraction=0.025, pad=0.02, label="features with ≥80% area coverage")
plt.show()

top_anomalies_df = regions_df.loc[regions_df["is_reviewable_anomaly"]].nlargest(25, "anomaly_score")[
    [
        "region_id",
        "scale",
        "analysis_cohort",
        "centroid_latitude",
        "centroid_longitude",
        "neutral_type",
        "rare_broad_type",
        "anomaly_score",
        "artifact_warning",
    ]
]
display(top_anomalies_df)


## Export reproducible region, neutral-signature, and evidence-audit artifacts


In [ ]:
def json_compatible(value: Any) -> Any:
    if value is pd.NA:
        return None
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


def labels_to_geojson(labels: np.ndarray, scale_name: str, region_id_lookup: dict[int, str], path: Path) -> None:
    cell_degrees = 1.0 if scale_name == "medium_1deg" else 10.0
    transform = from_origin(-180.0, 90.0, cell_degrees, cell_degrees)
    geometry_parts: defaultdict[int, list[list[Any]]] = defaultdict(list)
    for geometry, value in raster_shapes(
        labels.astype(np.int32),
        mask=labels > 0,
        transform=transform,
        connectivity=4,
    ):
        numeric_id = int(value)
        if geometry["type"] == "Polygon":
            geometry_parts[numeric_id].append(geometry["coordinates"])
        elif geometry["type"] == "MultiPolygon":
            geometry_parts[numeric_id].extend(geometry["coordinates"])
        else:
            raise ValueError(f"Unexpected raster polygon geometry: {geometry['type']}")
    features = []
    for numeric_id, polygons in geometry_parts.items():
        geometry = (
            {"type": "Polygon", "coordinates": polygons[0]}
            if len(polygons) == 1
            else {"type": "MultiPolygon", "coordinates": polygons}
        )
        features.append(
            {
                "type": "Feature",
                "properties": {"region_id": region_id_lookup[numeric_id], "scale": scale_name},
                "geometry": geometry,
            }
        )
    collection = {"type": "FeatureCollection", "features": features}
    path.write_text(json.dumps(collection, allow_nan=False, default=json_compatible), encoding="utf-8")


region_cells_export_columns = [
    "cell_id",
    "scale",
    "row",
    "column",
    "latitude",
    "longitude",
    "region_id",
    "broad_type",
    "neutral_type",
    "selected_algorithm",
    "cluster_confidence",
    "broad_cluster_confidence",
    "subtype_cluster_confidence",
    "limited_feature_evidence",
    "coverage_fragmented",
    "model_training_eligible",
    "cluster_assignment_role",
    "rare_broad_type",
    "artifact_diagnostic_available",
    "artifact_warning",
    "anomaly_score",
    "is_anomaly_candidate",
    "is_reviewable_anomaly",
]
region_cell_results_df[region_cells_export_columns].to_parquet(REGION_DIR / "mars_region_cells.parquet", index=False, compression="zstd")
regions_df.to_parquet(REGION_DIR / "mars_regions.parquet", index=False, compression="zstd")
adjacency_df.to_parquet(REGION_DIR / "mars_region_adjacency.parquet", index=False, compression="zstd")
cluster_summary_df.to_parquet(REGION_DIR / "mars_neutral_type_summary.parquet", index=False, compression="zstd")
cluster_profiles_df.to_parquet(REGION_DIR / "mars_neutral_type_profiles.parquet", index=False, compression="zstd")
region_feature_distributions_df.to_parquet(REGION_DIR / "mars_region_feature_distributions.parquet", index=False, compression="zstd")
neutral_type_distributions_df.to_parquet(REGION_DIR / "mars_neutral_type_distributions.parquet", index=False, compression="zstd")
neutral_type_dependencies_df.to_parquet(REGION_DIR / "mars_neutral_type_dependencies.parquet", index=False, compression="zstd")
neutral_type_spatial_signatures_df.to_parquet(REGION_DIR / "mars_neutral_type_spatial_signatures.parquet", index=False, compression="zstd")
top_anomalies_df.to_csv(REPORT_DIR / "top_anomaly_candidates.csv", index=False)
control_sites_df.to_csv(REPORT_DIR / "reference_locations.csv", index=False)
process_hypothesis_audit_df.to_csv(REPORT_DIR / "process_hypothesis_evidence_audit.csv", index=False)
model_comparison_df.to_csv(REPORT_DIR / "cluster_model_comparison.csv", index=False)
artifact_diagnostics_df.to_csv(REPORT_DIR / "coverage_artifact_diagnostics.csv", index=False)
artifact_importance_df.to_csv(REPORT_DIR / "coverage_artifact_feature_importance.csv", index=False)
ablation_df.to_csv(REPORT_DIR / "feature_group_ablation.csv", index=False)
quality_correction_df.to_csv(REPORT_DIR / "quality_effect_correction.csv", index=False)

(REPORT_DIR / "partition_manifest.json").write_text(
    json.dumps(partition_manifest, indent=2, allow_nan=False, default=json_compatible),
    encoding="utf-8",
)
(REPORT_DIR / "process_hypothesis_registry.json").write_text(
    json.dumps(PROCESS_HYPOTHESES, indent=2, allow_nan=False, default=json_compatible),
    encoding="utf-8",
)

np.savez_compressed(
    REGION_DIR / "mars_region_label_rasters.npz",
    medium_1deg=scale_label_rasters["medium_1deg"],
    coarse_10deg=scale_label_rasters["coarse_10deg"],
)

for scale_name, labels in scale_label_rasters.items():
    lookup = (
        regions_df.loc[regions_df["scale"] == scale_name]
        .set_index("region_numeric_id")["region_id"]
        .to_dict()
    )
    labels_to_geojson(labels, scale_name, lookup, REGION_DIR / f"mars_{scale_name}_regions.geojson")

for partition_id, model in segmentation_models.items():
    joblib.dump(model, MODEL_DIR / f"segmentation_{re.sub(r'[^a-zA-Z0-9_-]+', '_', partition_id)}.joblib")
for partition_id, model in selected_models.items():
    joblib.dump(model, MODEL_DIR / f"clustering_{re.sub(r'[^a-zA-Z0-9_-]+', '_', partition_id)}.joblib")
for partition_id, correction in quality_correction_models.items():
    joblib.dump(correction, MODEL_DIR / f"quality_correction_{re.sub(r'[^a-zA-Z0-9_-]+', '_', partition_id)}.joblib")
for partition_id, model in anomaly_models.items():
    joblib.dump(model, MODEL_DIR / f"anomaly_{re.sub(r'[^a-zA-Z0-9_-]+', '_', partition_id)}.joblib")

artifact_warning_partitions = (
    artifact_diagnostics_df.loc[artifact_diagnostics_df["artifact_warning"], "partition_id"].tolist()
    if len(artifact_diagnostics_df)
    else []
)
artifact_untested_partitions = sorted(
    set(partition_cluster_context) - set(artifact_available_partitions)
)
coverage_fragmented_partitions = partition_manifest_df.loc[
    partition_manifest_df["coverage_fragmented"], "partition_id"
].tolist()
fragmentation_warning_partitions = (
    region_summary.loc[
        (
            (region_summary["tiny_cell_percent"] > 100.0 * CFG.maximum_posthoc_coverage_fraction)
            | (region_summary["tiny_area_percent"] > 100.0 * CFG.maximum_posthoc_coverage_fraction)
        )
        & ~(region_summary["scale"] + "__" + region_summary["analysis_cohort"].astype(str)).isin(coverage_fragmented_partitions)
    ]
    .assign(partition_id=lambda frame: frame["scale"] + "__" + frame["analysis_cohort"].astype(str))["partition_id"]
    .tolist()
)
broad_discovery_ready = bool(
    not fragmentation_warning_partitions
    and not artifact_warning_partitions
    and not artifact_untested_partitions
    and len(neutral_type_distributions_df) > 0
)

# Intentionally false: this notebook has no independently validated process examples.
# Neutral distributions must not be handed to Earth matching as process distributions.
validated_process_signature_count = 0
earth_filter_handoff_ready = False
subtype_decision_rows = model_comparison_df.loc[
    (model_comparison_df["stage"] == "subtype") & model_comparison_df["best_candidate"]
] if len(model_comparison_df) else pd.DataFrame()
accepted_subtype_splits = int(subtype_decision_rows["applied_to_output"].sum()) if len(subtype_decision_rows) else 0
rejected_subtype_splits = int((~subtype_decision_rows["applied_to_output"]).sum()) if len(subtype_decision_rows) else 0
summary = {
    "status": "complete_broad_discovery" if broad_discovery_ready else "complete_needs_model_review",
    "run_started_utc": RUN_STARTED_UTC,
    "run_finished_utc": datetime.now(timezone.utc).isoformat(),
    "configuration": asdict(CFG),
    "input_root": str(INPUT_ROOT),
    "preparation_status": preparation_summary.get("status"),
    "input_cells": int(len(cells_df)),
    "connected_regions": int(len(regions_df)),
    "connected_regions_by_scale": regions_df.groupby("scale", observed=True)["region_id"].nunique().astype(int).to_dict(),
    "model_training_regions": int(sum(np.asarray(context["model_training_mask"], dtype=bool).sum() for context in partition_cluster_context.values())),
    "posthoc_tiny_regions": int((regions_df["cluster_assignment_role"] == "posthoc_tiny").sum()),
    "rare_broad_regions": int(regions_df["rare_broad_type"].sum()),
    "posthoc_coverage_by_partition": region_summary.set_index(
        region_summary["scale"] + "__" + region_summary["analysis_cohort"].astype(str)
    )[["tiny_region_percent", "tiny_cell_percent", "tiny_area_percent"]].to_dict(orient="index"),
    "neutral_types": int(regions_df["neutral_type"].nunique()),
    "neutral_types_by_partition": regions_df.groupby("partition_id", observed=True)["neutral_type"].nunique().astype(int).to_dict(),
    "neutral_distribution_rows": int(len(neutral_type_distributions_df)),
    "neutral_dependency_rows": int(len(neutral_type_dependencies_df)),
    "anomaly_candidates": int(regions_df["is_anomaly_candidate"].sum()),
    "reviewable_anomaly_candidates": int(regions_df["is_reviewable_anomaly"].sum()),
    "process_hypotheses_audited": int(len(process_hypothesis_audit_df)),
    "validated_process_signatures": validated_process_signature_count,
    "accepted_subtype_splits": accepted_subtype_splits,
    "rejected_subtype_splits": rejected_subtype_splits,
    "artifact_warning_partitions": artifact_warning_partitions,
    "artifact_untested_partitions": artifact_untested_partitions,
    "coverage_fragmented_partitions": coverage_fragmented_partitions,
    "fragmentation_warning_partitions": fragmentation_warning_partitions,
    "broad_discovery_ready": broad_discovery_ready,
    "earth_filter_handoff_ready": earth_filter_handoff_ready,
    "important_limits": [
        "Neutral clusters and their distributions are recurring property combinations, not proven causes.",
        "No process label is assigned from a hand-chosen percentile threshold.",
        "Marginal distributions alone are insufficient; dependence and spatial structure are exported separately.",
        "Static global products cannot establish present activity.",
        "Hydrogen-sensitive neutron evidence does not uniquely identify water ice.",
        "Known locations are sanity checks, not training labels or process validation.",
        "Intrinsically fragmented coverage is preserved as descriptive-only evidence and is not clustered.",
        "Tiny fragments do not influence model fitting; they are assigned afterward and explicitly flagged.",
        "Broad clusters smaller than the configured minimum are retained as rare candidates, not treated as recurring environments.",
        "Targeted data and multiple positive and negative controls are required before constructing a process signature.",
    ],
    "outputs": {
        "region_cells": str(REGION_DIR / "mars_region_cells.parquet"),
        "regions": str(REGION_DIR / "mars_regions.parquet"),
        "adjacency": str(REGION_DIR / "mars_region_adjacency.parquet"),
        "medium_geojson": str(REGION_DIR / "mars_medium_1deg_regions.geojson"),
        "coarse_geojson": str(REGION_DIR / "mars_coarse_10deg_regions.geojson"),
        "neutral_type_summary": str(REGION_DIR / "mars_neutral_type_summary.parquet"),
        "region_feature_distributions": str(REGION_DIR / "mars_region_feature_distributions.parquet"),
        "neutral_type_distributions": str(REGION_DIR / "mars_neutral_type_distributions.parquet"),
        "neutral_type_dependencies": str(REGION_DIR / "mars_neutral_type_dependencies.parquet"),
        "neutral_type_spatial_signatures": str(REGION_DIR / "mars_neutral_type_spatial_signatures.parquet"),
        "model_comparison": str(REPORT_DIR / "cluster_model_comparison.csv"),
        "artifact_diagnostics": str(REPORT_DIR / "coverage_artifact_diagnostics.csv"),
        "quality_correction": str(REPORT_DIR / "quality_effect_correction.csv"),
        "ablation": str(REPORT_DIR / "feature_group_ablation.csv"),
        "reference_locations": str(REPORT_DIR / "reference_locations.csv"),
        "process_hypothesis_audit": str(REPORT_DIR / "process_hypothesis_evidence_audit.csv"),
        "process_hypothesis_registry": str(REPORT_DIR / "process_hypothesis_registry.json"),
    },
}
(REPORT_DIR / "run_summary.json").write_text(
    json.dumps(summary, indent=2, allow_nan=False, default=json_compatible),
    encoding="utf-8",
)

summary_markdown = f"""# Mars region-discovery run summary

- Status: **{summary['status']}**
- Input cells: **{summary['input_cells']:,}**
- Connected regions: **{summary['connected_regions']:,}**
- Neutral types: **{summary['neutral_types']}**
- Neutral marginal-distribution rows: **{summary['neutral_distribution_rows']:,}**
- Neutral feature-dependency rows: **{summary['neutral_dependency_rows']:,}**
- Reviewable unusual-region candidates: **{summary['reviewable_anomaly_candidates']:,}**
- Process hypotheses audited against available evidence: **{summary['process_hypotheses_audited']}**
- Independently validated process signatures: **{summary['validated_process_signatures']}**
- Accepted rich-measurement subtype splits: **{summary['accepted_subtype_splits']}**
- Rejected weak, unstable, artifact-risk, or untestable subtype splits: **{summary['rejected_subtype_splits']}**
- Partitions with a coverage-artifact warning: **{len(summary['artifact_warning_partitions'])}**
- Clustered partitions without a usable artifact test: **{len(summary['artifact_untested_partitions'])}**
- Descriptive-only partitions with intrinsically fragmented coverage: **{len(summary['coverage_fragmented_partitions'])}**
- Partitions where post-hoc tiny fragments exceed the allowed cell/area coverage: **{len(summary['fragmentation_warning_partitions'])}**
- Broad neutral discovery ready for scientific review: **{summary['broad_discovery_ready']}**
- Ready for Earth process matching: **{summary['earth_filter_handoff_ready']}**

## Correct interpretation

The output contains connected regions, recurring neutral types, anomalies, marginal distributions, feature dependencies, spatial summaries, and a literature evidence audit. It contains **no validated process distribution yet**. The next stage must validate multiple examples with the missing evidence listed in the process registry before any Mars-to-Earth process match is calculated.
"""
(REPORT_DIR / "run_summary.md").write_text(summary_markdown, encoding="utf-8")
display(Markdown(summary_markdown))
display(pd.DataFrame([{"artifact": name, "path": path} for name, path in summary["outputs"].items()]))


## Handoff

Do **not** use neutral-type distributions as Earth filters. First review segmentation, stability, artifact warnings, anomalies, and signature coverage. Then retrieve the targeted observations named in the process registry for several positive examples and negative controls.

Only after those examples pass paper-defined evidence requirements should we aggregate their harmonized measurements into a validated process signature containing marginal distributions, feature dependencies, spatial organization, scale, and uncertainty. Earth regions can then be ranked by distribution distance while preserving where the analogy breaks down.
